
# NexusMart Datathon — Data Cleaning and Validation

## Objective
Clean the supplied datasets using the NexusMart challenge requirements.
All transformations will be implemented in code, with cleaning decisions
and before-and-after counts recorded. The original CSV files will remain
unchanged, and records that cannot be repaired will be preserved with
documented rejection reasons.


In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
REJECT_DIR = DATA_DIR / "rejected"
REPORT_DIR = PROJECT_ROOT / "reports"

for folder in [PROCESSED_DIR, REJECT_DIR, REPORT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Raw CSV files:", len(list(DATA_DIR.glob("*.csv"))))
print("Processed output:", PROCESSED_DIR)
print("Rejected records:", REJECT_DIR)

Project: c:\Users\LENOVO\Documents\CodeConnect26_DataAnalytics
Raw CSV files: 9
Processed output: c:\Users\LENOVO\Documents\CodeConnect26_DataAnalytics\data\processed
Rejected records: c:\Users\LENOVO\Documents\CodeConnect26_DataAnalytics\data\rejected



## 1. Baseline Counts

Record the row counts before cleaning. These values will be compared with
the cleaned and rejected records so that changes can be explained.


In [2]:
baseline = []

for file_path in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)

    baseline.append({
        "file": file_path.name,
        "rows_before": len(df),
        "columns_before": len(df.columns),
        "exact_duplicates_before": int(df.duplicated().sum()),
    })

baseline_df = pd.DataFrame(baseline)
display(baseline_df)

baseline_df.to_csv(
    REPORT_DIR / "cleaning_baseline.csv",
    index=False
)

,file,rows_before,columns_before,exact_duplicates_before
0,customers.csv,45000,8,0
1,locations.csv,650,7,0
2,order_items.csv,315143,7,0
3,orders.csv,163200,8,3200
4,payments.csv,168111,6,3296
5,products.csv,3200,9,0
6,returns.csv,16048,6,0
7,sellers.csv,420,6,0
8,shipping.csv,151228,7,0



## 2. Remove Exact Duplicate Rows

The initial profiling identified exact duplicate rows in the orders and
payments datasets. This step removes only completely identical rows from
the working copies. The raw files remain unchanged, and the number of
removed rows is recorded for each dataset.

Repeated identifiers with different values will be investigated separately.


In [3]:
duplicate_log = []

for file_path in sorted(DATA_DIR.glob("*.csv")):
    df = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)

    rows_before = len(df)
    duplicate_count = int(df.duplicated().sum())

    # Keep the first occurrence of each completely identical row.
    df_clean = df.drop_duplicates().copy()

    rows_after = len(df_clean)

    # Save the working copy; never overwrite the raw CSV.
    output_path = PROCESSED_DIR / file_path.name
    df_clean.to_csv(output_path, index=False)

    duplicate_log.append({
        "file": file_path.name,
        "rows_before": rows_before,
        "exact_duplicates_removed": duplicate_count,
        "rows_after": rows_after,
        "rows_accounted_for": rows_after + duplicate_count,
    })

duplicate_log_df = pd.DataFrame(duplicate_log)
display(duplicate_log_df)

duplicate_log_df.to_csv(
    REPORT_DIR / "duplicate_removal_log.csv",
    index=False
)

,file,rows_before,exact_duplicates_removed,rows_after,rows_accounted_for
0,customers.csv,45000,0,45000,45000
1,locations.csv,650,0,650,650
2,order_items.csv,315143,0,315143,315143
3,orders.csv,163200,3200,160000,163200
4,payments.csv,168111,3296,164815,168111
5,products.csv,3200,0,3200,3200
6,returns.csv,16048,0,16048,16048
7,sellers.csv,420,0,420,420
8,shipping.csv,151228,0,151228,151228



## 3. Validate Duplicate Removal

Confirm that the processed files contain no remaining exact duplicate rows
and that the before-and-after row counts reconcile.


In [4]:
duplicate_validation = []

for file_path in sorted(PROCESSED_DIR.glob("*.csv")):
    df = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)

    duplicate_validation.append({
        "file": file_path.name,
        "rows_after_cleaning": len(df),
        "remaining_exact_duplicates": int(df.duplicated().sum()),
    })

duplicate_validation_df = pd.DataFrame(duplicate_validation)
display(duplicate_validation_df)

print(
    "No exact duplicates remain:",
    duplicate_validation_df["remaining_exact_duplicates"].eq(0).all()
)

print(
    "Row counts reconcile:",
    (
        duplicate_log_df["rows_accounted_for"]
        == duplicate_log_df["rows_before"]
    ).all()
)

,file,rows_after_cleaning,remaining_exact_duplicates
0,customers.csv,45000,0
1,locations.csv,650,0
2,order_items.csv,315143,0
3,orders.csv,160000,0
4,payments.csv,164815,0
5,products.csv,3200,0
6,returns.csv,16048,0
7,sellers.csv,420,0
8,shipping.csv,151228,0


No exact duplicates remain: True
Row counts reconcile: True


## 4. Missing-Value Audit

Identify missing values in each processed dataset before deciding how to handle them. Missing values will be treated according to their business meaning: some may be safely imputed, some may be flagged or retained, and others may require quarantine. No missing value will be filled without a documented reason.

In [5]:

missing_audit = []

for file_path in sorted(PROCESSED_DIR.glob("*.csv")):
    df = pd.read_csv(file_path, encoding="utf-8-sig", low_memory=False)

    for column in df.columns:
        missing_count = int(df[column].isna().sum())

        if missing_count > 0:
            missing_audit.append({
                "file": file_path.name,
                "column": column,
                "missing_count": missing_count,
                "missing_percent": round(
                    missing_count / len(df) * 100, 2
                ) if len(df) else 0
            })

missing_audit_df = pd.DataFrame(missing_audit)
missing_audit_df = missing_audit_df.sort_values(
    ["file", "missing_count"],
    ascending=[True, False]
)

display(missing_audit_df)

missing_audit_df.to_csv(
    REPORT_DIR / "missing_value_audit.csv",
    index=False
)


,file,column,missing_count,missing_percent
4,customers.csv,birth_year,4600,10.22
1,customers.csv,phone,3599,8.00
3,customers.csv,city_id,2233,4.96
2,customers.csv,gender,2199,4.89
0,customers.csv,email,1698,3.77
6,locations.csv,latitude,33,5.08
7,locations.csv,longitude,31,4.77
5,locations.csv,province,25,3.85
8,order_items.csv,discount_amount,31567,10.02
10,orders.csv,discount_code,109871,68.67


## 5. Date-Quality Investigation

Inspect date strings that cannot be parsed and order dates outside the permitted period of 1 January 2024 through 30 June 2025. Dates will be repaired only when supported by other data; unresolved records will be preserved with a rejection reason.

### 5.1 Audit Date Columns

Inspect date fields across orders, payments, shipping, returns and sellers. Identify missing values, unparseable date strings, and dates that require further investigation. This step profiles the data without modifying it.

In [6]:

date_columns = {
    "orders.csv": ["order_date"],
    "payments.csv": ["payment_date"],
    "shipping.csv": ["shipped_date", "delivered_date"],
    "returns.csv": ["return_date"],
    "sellers.csv": ["join_date"],
}

date_investigation = []

for filename, columns in date_columns.items():
    df = pd.read_csv(
        PROCESSED_DIR / filename,
        encoding="utf-8-sig",
        low_memory=False,
        dtype=str
    )

    for column in columns:
        if column not in df.columns:
            continue

        raw = df[column].str.strip()

        # First-pass parser for investigation only.
        parsed = pd.to_datetime(
            raw,
            format="mixed",
            dayfirst=True,
            errors="coerce"
        )

        invalid = raw.notna() & raw.ne("") & parsed.isna()

        print(f"\n{filename} — {column}")
        print("Missing values:", int(raw.isna().sum() + raw.eq("").sum()))
        print("Unparseable non-empty values:", int(invalid.sum()))

        if invalid.any():
            display(
                df.loc[invalid, [column]].head(15)
            )

        if filename == "orders.csv" and column == "order_date":
            out_of_range = (
                parsed.notna()
                & (
                    (parsed < pd.Timestamp("2024-01-01"))
                    | (parsed > pd.Timestamp("2025-06-30"))
                )
            )

            print("Parsed dates outside allowed order period:",
                  int(out_of_range.sum()))

            if out_of_range.any():
                display(
                    df.loc[out_of_range].head(15)
                )



orders.csv — order_date
Missing values: 0
Unparseable non-empty values: 890


,order_date
52,00/00/2024
80,00/00/2024
579,2024-02-30
770,2025-04-31
969,2024-04-31
1166,2024-04-31
1591,2024-04-31
1711,00/00/2024
1752,2024-02-30
1780,2024-04-31


Parsed dates outside allowed order period: 481


,order_id,customer_id,order_date,order_status,ship_city_id,order_total,discount_code,channel
325,O0037450,C005260,2026-08-21,Delivered,535.0,3395,NaN,Web
757,O0012067,C019909,2026-05-14,Returned,347.0,2664,NaN,App
985,O0100747,C020070,2027-03-27,delivered,281.0,16448,save40,APP
1124,O0076202,C032846,2027-01-10,Delivered,450.0,32004,NaN,Web
1290,O0107005,C034957,2027-04-17,Delivered,291.0,1229,NaN,App
2246,O0085364,C031787,2027-02-06,Delivered,201.0,4366,NaN,Web
2867,O0050234,C043492,09/10/2026,DELIVERD,155.0,2970,NaN,Mobile Web
3077,O0059080,C029768,Nov 9 2026,Delivered,151.0,1000,NaN,Web
3418,O0125991,C033100,2027-06-06,Delivered,198.0,5664,NaN,App
3448,O0017518,C023930,2026-06-05,Delivered,281.0,1849,SAVE50,Mobile Web



payments.csv — payment_date
Missing values: 4927
Unparseable non-empty values: 857


...

,delivered_date
226,2024-04-31
440,2024-04-31
706,00/00/2024
1057,2024-04-31
1312,2024-02-30
1559,2024-04-31
1600,2024-04-31
1771,00/00/2024
1994,2024-02-30
2014,2024-02-30



returns.csv — return_date
Missing values: 0
Unparseable non-empty values: 106


,return_date
15,00/00/2024
53,2024-02-30
97,2024-02-30
665,2024-02-30
787,00/00/2024
818,2024-02-30
868,2024-06-31
1031,00/00/2024
1646,2024-02-30
1803,2024-04-31



sellers.csv — join_date
Missing values: 34
Unparseable non-empty values: 4


,join_date
69,2020-06-31
112,2020-06-31
146,00/00/2021
221,2022-04-31


### 5.2 Record Cleaning Decisions

Create an initial log of major data-quality issues, their proposed handling rules, business reasons, and current status. Decisions remain pending until the relevant data has been investigated and validated.

In [7]:

cleaning_decisions = pd.DataFrame([
    {
        "issue": "Exact duplicate rows",
        "decision": "Keep first identical row",
        "business_reason": "Exact copies do not represent additional distinct records",
        "status": "Completed",
    },
    {
        "issue": "Missing values",
        "decision": "Decide per column after investigation",
        "business_reason": "Missing fields have different business meanings",
        "status": "Investigation required",
    },
    {
        "issue": "Invalid dates",
        "decision": "Repair with supporting evidence or quarantine",
        "business_reason": "Invalid dates can distort reporting and time analysis",
        "status": "Investigation required",
    },
    {
        "issue": "Broken references",
        "decision": "Map to Unknown or quarantine using a documented rule",
        "business_reason": "Preserve referential integrity in the warehouse",
        "status": "Not started",
    },
])

display(cleaning_decisions)

cleaning_decisions.to_csv(
    REPORT_DIR / "cleaning_decision_log.csv",
    index=False
)


,issue,decision,business_reason,status
0,Exact duplicate rows,Keep first identical row,Exact copies do not represent additional disti...,Completed
1,Missing values,Decide per column after investigation,Missing fields have different business meanings,Investigation required
2,Invalid dates,Repair with supporting evidence or quarantine,Invalid dates can distort reporting and time a...,Investigation required
3,Broken references,Map to Unknown or quarantine using a documente...,Preserve referential integrity in the warehouse,Not started


### 5.3 Investigate the Allowed Order-Date Period

Check order dates against the challenge's permitted period of 1 January 2024 through 30 June 2025. Separate impossible or unparseable dates from valid dates outside the allowed period, and record the number of affected rows.

In [8]:

orders = pd.read_csv(
    PROCESSED_DIR / "orders.csv",
    encoding="utf-8-sig",
    low_memory=False,
    dtype=str
)

raw_dates = orders["order_date"].str.strip()

parsed_dates = pd.to_datetime(
    raw_dates,
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

invalid_dates = raw_dates.notna() & raw_dates.ne("") & parsed_dates.isna()

outside_period = (
    parsed_dates.notna()
    & (
        (parsed_dates < pd.Timestamp("2024-01-01"))
        | (parsed_dates > pd.Timestamp("2025-06-30"))
    )
)

print("Processed orders:", len(orders))
print("Unparseable non-empty dates:", int(invalid_dates.sum()))
print("Valid dates outside allowed period:", int(outside_period.sum()))
print("Total rows requiring date investigation:",
      int((invalid_dates | outside_period).sum()))
print("Overlap between these two groups:",
      int((invalid_dates & outside_period).sum()))

print("\nMost common invalid date strings:")
display(
    raw_dates[invalid_dates]
    .value_counts()
    .head(20)
    .rename_axis("raw_date")
    .reset_index(name="count")
)

print("\nExamples of unparseable orders:")
display(
    orders.loc[
        invalid_dates,
        ["order_id", "customer_id", "order_date", "order_status", "order_total"]
    ].head(20)
)

print("\nExamples of out-of-range orders:")
display(
    orders.loc[
        outside_period,
        ["order_id", "customer_id", "order_date", "order_status", "order_total"]
    ].head(20)
)


Processed orders: 160000
Unparseable non-empty dates: 890
Valid dates outside allowed period: 481
Total rows requiring date investigation: 1371
Overlap between these two groups: 0

Most common invalid date strings:


,raw_date,count
0,00/00/2024,146
1,2024-06-31,135
2,2024-04-31,129
3,2024-02-30,121
4,2025-04-31,97
5,00/00/2025,91
6,2025-02-30,85
7,2025-06-31,84
8,2026-04-31,1
9,2027-02-30,1



Examples of unparseable orders:


,order_id,customer_id,order_date,order_status,order_total
52,O0088204,C001641,00/00/2024,delivered,6106
80,O0065554,C016450,00/00/2024,Delivered,229
579,O0024244,C024476,2024-02-30,Delivered,2219
770,O0128817,C031276,2025-04-31,Delivered,1100
969,O0019440,C016936,2024-04-31,Delivered,4514
1166,O0003688,C001315,2024-04-31,Delivered,5689
1591,O0007046,C003365,2024-04-31,Delivered,1364
1711,O0096131,C043966,00/00/2024,Delivered,5994
1752,O0066614,C000523,2024-02-30,Completed,4251
1780,O0053941,C013838,2024-04-31,Returned,3086



Examples of out-of-range orders:


,order_id,customer_id,order_date,order_status,order_total
325,O0037450,C005260,2026-08-21,Delivered,3395
757,O0012067,C019909,2026-05-14,Returned,2664
985,O0100747,C020070,2027-03-27,delivered,16448
1124,O0076202,C032846,2027-01-10,Delivered,32004
1290,O0107005,C034957,2027-04-17,Delivered,1229
2246,O0085364,C031787,2027-02-06,Delivered,4366
2867,O0050234,C043492,09/10/2026,DELIVERD,2970
3077,O0059080,C029768,Nov 9 2026,Delivered,1000
3418,O0125991,C033100,2027-06-06,Delivered,5664
3448,O0017518,C023930,2026-06-05,Delivered,1849


### 5.4 Cross-Check Suspicious Orders Against Related Records

Identify orders with suspicious dates and inspect whether corresponding payment, shipment, delivery and order-item records exist. These relationships help assess data consistency but do not, by themselves, establish the correct order date.

In [9]:

suspect_ids = set(
    orders.loc[invalid_dates | outside_period, "order_id"].dropna()
)

payments = pd.read_csv(
    PROCESSED_DIR / "payments.csv",
    encoding="utf-8-sig",
    low_memory=False,
    dtype=str
)

shipping = pd.read_csv(
    PROCESSED_DIR / "shipping.csv",
    encoding="utf-8-sig",
    low_memory=False,
    dtype=str
)

items = pd.read_csv(
    PROCESSED_DIR / "order_items.csv",
    encoding="utf-8-sig",
    low_memory=False,
    dtype=str
)

print("Suspicious order IDs:", len(suspect_ids))
print("Matching order IDs in payments:",
      payments["order_id"].isin(suspect_ids).sum()
      if "order_id" in payments.columns else "No order_id column")

print("Matching order IDs in shipping:",
      shipping["order_id"].isin(suspect_ids).sum()
      if "order_id" in shipping.columns else "No order_id column")

print("Matching order IDs in order items:",
      items["order_id"].isin(suspect_ids).sum()
      if "order_id" in items.columns else "No order_id column")

if "order_id" in payments.columns:
    display(
        payments.loc[
            payments["order_id"].isin(suspect_ids),
            ["order_id", "payment_date"]
        ].head(15)
    )

if "order_id" in shipping.columns:
    display(
        shipping.loc[
            shipping["order_id"].isin(suspect_ids),
            ["order_id", "shipped_date", "delivered_date"]
        ].head(15)
    )


Suspicious order IDs: 1371
Matching order IDs in payments: 1412
Matching order IDs in shipping: 1304
Matching order IDs in order items: 2718


,order_id,payment_date
62,O0041667,Jun 30 2024
164,O0050254,2024-07-31
476,O0109607,2025-02-19
589,O0127007,2025-04-03
636,O0089141,2024-12-20
925,O0040923,2024-06-25
929,O0049748,2024-08-01
949,O0145415,2025-05-26
1061,O0155069,2025-06-16
1103,O0004882,2024-01-28


,order_id,shipped_date,delivered_date
19,O0000020,2024-01-02,NaN
144,O0000149,2024-01-01,2024-01-05
154,O0000160,2024-01-02,2024-01-03
224,O0000233,2024-01-03,2024-01-04
228,O0000237,2024-01-03,08/01/2024
254,O0000263,2024-01-04,2024-01-10
413,O0000430,2024-01-04,2024-01-06
841,O0000872,06/01/2024,2024-01-07
912,O0000947,2024-01-06,2024-01-10
916,O0000951,2024-01-08,Jan 9 2024


In [10]:

# Reuse the orders, payments, shipping, parsed_dates,
# invalid_dates, and outside_period variables from Step 18.

order_audit = orders[[
    "order_id", "order_date", "order_status", "order_total"
]].copy()

order_audit["parsed_order_date"] = parsed_dates
order_audit["date_issue"] = ""

order_audit.loc[invalid_dates, "date_issue"] = "INVALID_DATE"
order_audit.loc[outside_period, "date_issue"] = "OUT_OF_RANGE"

payment_audit = payments.copy()
shipping_audit = shipping.copy()

# Normalize IDs for matching without changing the source files.
for frame in [order_audit, payment_audit, shipping_audit]:
    frame["order_id"] = frame["order_id"].str.strip()

payment_audit["payment_date_parsed"] = pd.to_datetime(
    payment_audit["payment_date"].str.strip(),
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

shipping_audit["shipped_date_parsed"] = pd.to_datetime(
    shipping_audit["shipped_date"].str.strip(),
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

shipping_audit["delivered_date_parsed"] = pd.to_datetime(
    shipping_audit["delivered_date"].str.strip(),
    format="mixed",
    dayfirst=True,
    errors="coerce"
)

payment_summary = (
    payment_audit.groupby("order_id", dropna=False)
    .agg(
        payment_rows=("order_id", "size"),
        earliest_payment=("payment_date_parsed", "min"),
        latest_payment=("payment_date_parsed", "max")
    )
    .reset_index()
)

shipping_summary = (
    shipping_audit.groupby("order_id", dropna=False)
    .agg(
        shipment_rows=("order_id", "size"),
        earliest_shipped=("shipped_date_parsed", "min"),
        latest_delivered=("delivered_date_parsed", "max")
    )
    .reset_index()
)

order_audit = order_audit.merge(
    payment_summary, on="order_id", how="left", validate="one_to_one"
)

order_audit = order_audit.merge(
    shipping_summary, on="order_id", how="left", validate="one_to_one"
)

suspect_audit = order_audit.loc[
    order_audit["date_issue"].ne("")
].copy()

display(suspect_audit.head(25))

suspect_audit.to_csv(
    REPORT_DIR / "suspicious_order_date_investigation.csv",
    index=False
)

print("Suspicious orders audited:", len(suspect_audit))
print("With at least one parseable payment date:",
      int(suspect_audit["earliest_payment"].notna().sum()))
print("With a parseable shipment date:",
      int(suspect_audit["earliest_shipped"].notna().sum()))
print("With a parseable delivery date:",
      int(suspect_audit["latest_delivered"].notna().sum()))


,order_id,order_date,order_status,order_total,parsed_order_date,date_issue,payment_rows,earliest_payment,latest_payment,shipment_rows,earliest_shipped,latest_delivered
52,O0088204,00/00/2024,delivered,6106,NaT,INVALID_DATE,1,2024-12-06,2024-12-06,1.0,2024-12-06,2024-12-10
80,O0065554,00/00/2024,Delivered,229,NaT,INVALID_DATE,1,2024-09-26,2024-09-26,1.0,2024-09-23,2024-09-26
325,O0037450,2026-08-21,Delivered,3395,2026-08-21,OUT_OF_RANGE,1,2024-06-16,2024-06-16,1.0,2024-06-13,2024-06-16
579,O0024244,2024-02-30,Delivered,2219,NaT,INVALID_DATE,1,NaT,NaT,1.0,2024-04-15,2024-04-19
757,O0012067,2026-05-14,Returned,2664,2026-05-14,OUT_OF_RANGE,1,2024-03-05,2024-03-05,1.0,2024-03-06,2024-03-07
770,O0128817,2025-04-31,Delivered,1100,NaT,INVALID_DATE,1,2025-04-03,2025-04-03,1.0,2025-04-04,2025-04-06
969,O0019440,2024-04-31,Delivered,4514,NaT,INVALID_DATE,1,2024-04-05,2024-04-05,1.0,2024-04-04,2024-04-05
985,O0100747,2027-03-27,delivered,16448,2027-03-27,OUT_OF_RANGE,1,2025-01-19,2025-01-19,1.0,2025-01-17,2025-01-19
1124,O0076202,2027-01-10,Delivered,32004,2027-01-10,OUT_OF_RANGE,1,2024-11-06,2024-11-06,1.0,NaT,2024-11-06
1166,O0003688,2024-04-31,Delivered,5689,NaT,INVALID_DATE,1,2024-01-21,2024-01-21,1.0,2024-01-21,NaT


Suspicious orders audited: 1371
With at least one parseable payment date: 1320
With a parseable shipment date: 1242
With a parseable delivery date: 1138


## 6. Missing-Value Handling Decisions

Document proposed actions for each column with missing values, including retention, flagging, investigation, imputation or quarantine where appropriate. Each decision must have a business justification. This register records proposed actions; it does not mean that the missing values have already been resolved.

In [11]:

missing_decisions = [
    # Customers
    ("customers.csv", "email", "Retain missing", "Contact detail is optional for analysis; do not invent it."),
    ("customers.csv", "phone", "Retain missing", "Contact detail is optional for analysis; do not invent it."),
    ("customers.csv", "gender", "Flag as Unknown", "Avoid assigning an unsupported demographic value."),
    ("customers.csv", "city_id", "Investigate reference; otherwise Unknown", "Customer geography is unavailable unless a valid mapping exists."),
    ("customers.csv", "birth_year", "Retain missing; flag for demographic analysis", "Do not infer age without reliable evidence."),

    # Locations
    ("locations.csv", "province", "Investigate; otherwise Unknown", "Do not infer province from city name alone."),
    ("locations.csv", "latitude", "Retain missing unless reliable mapping exists", "Do not fabricate coordinates."),
    ("locations.csv", "longitude", "Retain missing unless reliable mapping exists", "Do not fabricate coordinates."),

    # Order items
    ("order_items.csv", "discount_amount", "Investigate numeric meaning; then resolve", "Missing may mean no discount, but confirm against order totals and business rules before imputing."),

    # Orders
    ("orders.csv", "discount_code", "Normalize to no discount", "The challenge explicitly states that an empty code means no discount."),
    ("orders.csv", "ship_city_id", "Investigate reference; otherwise Unknown", "Shipping geography should not be guessed."),

    # Payments
    ("payments.csv", "payment_date", "Retain missing; investigate payment status", "A missing date does not establish when payment occurred."),

    # Products
    ("products.csv", "sub_category", "Flag as Unknown", "Preserve the product while acknowledging missing classification."),
    ("products.csv", "brand", "Flag as Unknown", "Do not invent a brand."),
    ("products.csv", "weight_kg", "Retain missing; flag for shipping analysis", "Do not invent product weight."),
    ("products.csv", "cost_price", "Retain missing; flag for margin calculations", "Missing cost must not be silently treated as zero."),

    # Returns
    ("returns.csv", "return_reason", "Flag as Unknown", "A valid return can exist without a recorded reason."),

    # Sellers
    ("sellers.csv", "join_date", "Repair if supported; otherwise quarantine or retain as unresolved", "Seller tenure cannot be calculated reliably from an invalid date."),
    ("sellers.csv", "rating", "Retain missing; flag for seller scoring", "Do not assume a missing rating is zero."),
    ("sellers.csv", "seller_tier", "Flag as Unknown", "Do not assign a tier without a supported rule."),

    # Shipping
    ("shipping.csv", "shipped_date", "Investigate shipping status; retain or quarantine if invalid", "Missing shipment date may be legitimate for unshipped orders."),
    ("shipping.csv", "delivered_date", "Investigate shipping status; retain or quarantine if invalid", "Missing delivery date may mean the order is not yet delivered."),
    ("shipping.csv", "shipping_fee", "Investigate against order/shipment data before imputation", "Shipping is a courier charge per order, not per order line.")
]

missing_decision_df = pd.DataFrame(
    missing_decisions,
    columns=["file", "column", "proposed_action", "business_reason"]
)

missing_decision_df["status"] = "Pending validation"

display(missing_decision_df)

missing_decision_df.to_csv(
    REPORT_DIR / "missing_value_decisions.csv",
    index=False
)


,file,column,proposed_action,business_reason,status
0,customers.csv,email,Retain missing,Contact detail is optional for analysis; do no...,Pending validation
1,customers.csv,phone,Retain missing,Contact detail is optional for analysis; do no...,Pending validation
2,customers.csv,gender,Flag as Unknown,Avoid assigning an unsupported demographic value.,Pending validation
3,customers.csv,city_id,Investigate reference; otherwise Unknown,Customer geography is unavailable unless a val...,Pending validation
4,customers.csv,birth_year,Retain missing; flag for demographic analysis,Do not infer age without reliable evidence.,Pending validation
5,locations.csv,province,Investigate; otherwise Unknown,Do not infer province from city name alone.,Pending validation
6,locations.csv,latitude,Retain missing unless reliable mapping exists,Do not fabricate coordinates.,Pending validation
7,locations.csv,longitude,Retain missing unless reliable mapping exists,Do not fabricate coordinates.,Pending validation
8,order_items.csv,discount_amount,Investigate numeric meaning; then resolve,"Missing may mean no discount, but confirm agai...",Pending validation
9,orders.csv,discount_code,Normalize to no discount,The challenge explicitly states that an empty ...,Pending validation


## 7. Inconsistent Category Standardization

Inspect categorical values for inconsistent capitalization, spelling and naming conventions. Record the observed variations before applying documented standardization rules. Do not merge distinct business statuses simply because their wording looks similar.

In [12]:

category_audit = []

for file_path in sorted(PROCESSED_DIR.glob("*.csv")):
    df = pd.read_csv(
        file_path, encoding="utf-8-sig",
        low_memory=False
    )

    for column in df.select_dtypes(include=["object", "string"]).columns:
        values = df[column].dropna().astype(str).str.strip()

        if values.empty:
            continue

        # Group case/spacing variations for investigation.
        normalized = values.str.lower()
        grouped = (
            pd.DataFrame({"original": values, "normalized": normalized})
            .groupby("normalized")["original"]
            .agg(lambda x: sorted(x.unique().tolist()))
        )

        for normalized_value, variants in grouped.items():
            if len(variants) > 1:
                category_audit.append({
                    "file": file_path.name,
                    "column": column,
                    "normalized_candidate": normalized_value,
                    "observed_variants": " | ".join(variants),
                    "variant_count": len(variants)
                })

category_audit_df = pd.DataFrame(category_audit)
display(category_audit_df)

category_audit_df.to_csv(
    REPORT_DIR / "category_variation_audit.csv",
    index=False
)

print("Potential case/spacing inconsistencies:",
      len(category_audit_df))


,file,column,normalized_candidate,observed_variants,variant_count
0,customers.csv,full_name,ahmed abbasi,AHMED ABBASI | Ahmed Abbasi,2
1,customers.csv,full_name,ahmed akhtar,Ahmed Akhtar | ahmed akhtar,2
2,customers.csv,full_name,ahmed ansari,Ahmed Ansari | ahmed ansari,2
3,customers.csv,full_name,ahmed baig,AHMED BAIG | Ahmed Baig,2
4,customers.csv,full_name,ahmed butt,Ahmed Butt | ahmed butt,2
...,...,...,...,...,...
2266,shipping.csv,courier,tcs,TCS | tcs,2
2267,shipping.csv,courier,trax,TRAX | Trax | trax,3
2268,shipping.csv,delivery_status,delivered,DELIVERED | Delivered | delivered,3
2269,shipping.csv,delivery_status,in transit,IN TRANSIT | In Transit | in transit,3


Potential case/spacing inconsistencies: 2271


## 8. Incorrect Data Types and Numeric Validation

Inspect monetary, quantity, weight, rating and coordinate fields for values that cannot be safely interpreted as numbers. Values containing currency symbols or separators will be identified before conversion. The original text values will be preserved for investigation.

In [13]:

numeric_candidates = {
    "orders.csv": ["order_total"],
    "order_items.csv": [
        "quantity", "unit_price", "discount_amount", "amount"
    ],
    "payments.csv": ["amount", "payment_amount"],
    "products.csv": ["cost_price", "weight_kg"],
    "returns.csv": ["refund_amount"],
    "shipping.csv": ["shipping_fee"],
    "sellers.csv": ["rating"],
    "locations.csv": ["latitude", "longitude"],
}

numeric_audit = []

for filename, columns in numeric_candidates.items():
    path = PROCESSED_DIR / filename
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)

    for column in columns:
        if column not in df.columns:
            continue

        raw = df[column].astype("string").str.strip()
        cleaned = raw.str.replace(",", "", regex=False)
        cleaned = cleaned.str.replace(r"(?i)^rs\.?\s*", "", regex=True)
        cleaned = cleaned.str.replace(r"^\$\s*", "", regex=True)

        numeric = pd.to_numeric(cleaned, errors="coerce")
        bad = raw.notna() & raw.ne("") & numeric.isna()

        numeric_audit.append({
            "file": filename,
            "column": column,
            "non_missing_values": int((raw.notna() & raw.ne("")).sum()),
            "unparseable_values": int(bad.sum()),
            "examples": " | ".join(raw[bad].head(5).tolist())
        })

numeric_audit_df = pd.DataFrame(numeric_audit)
display(numeric_audit_df)

numeric_audit_df.to_csv(
    REPORT_DIR / "numeric_type_audit.csv",
    index=False
)


,file,column,non_missing_values,unparseable_values,examples
0,orders.csv,order_total,160000,0,
1,order_items.csv,quantity,315143,965,one | one | three | four | one
2,order_items.csv,unit_price,315143,0,
3,order_items.csv,discount_amount,283576,0,
4,payments.csv,amount,164815,0,
5,products.csv,cost_price,3030,0,
6,products.csv,weight_kg,2904,0,
7,returns.csv,refund_amount,16048,0,
8,shipping.csv,shipping_fee,142083,0,
9,sellers.csv,rating,387,0,


## 9. Outlier Investigation

Investigate implausible numeric values using domain-based limits and distribution summaries. Statistical extremes will be flagged for review rather than automatically deleted, since unusually large orders or refunds may be legitimate.

In [14]:

outlier_candidates = {
    "orders.csv": ["order_total"],
    "order_items.csv": ["quantity", "unit_price", "discount_amount", "amount"],
    "products.csv": ["cost_price", "weight_kg"],
    "returns.csv": ["refund_amount"],
    "shipping.csv": ["shipping_fee"],
    "sellers.csv": ["rating"],
}

outlier_audit = []

for filename, columns in outlier_candidates.items():
    df = pd.read_csv(
        PROCESSED_DIR / filename,
        encoding="utf-8-sig",
        low_memory=False
    )

    for column in columns:
        if column not in df.columns:
            continue

        raw = (
            df[column].astype("string").str.strip()
            .str.replace(",", "", regex=False)
            .str.replace(r"(?i)^rs\.?\s*", "", regex=True)
        )
        values = pd.to_numeric(raw, errors="coerce").dropna()

        if values.empty:
            continue

        q1 = values.quantile(0.25)
        q3 = values.quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr

        outlier_audit.append({
            "file": filename,
            "column": column,
            "valid_numeric_count": len(values),
            "minimum": values.min(),
            "maximum": values.max(),
            "negative_count": int((values < 0).sum()),
            "iqr_lower_bound": lower,
            "iqr_upper_bound": upper,
            "statistical_outlier_count": int(
                ((values < lower) | (values > upper)).sum()
            )
        })

outlier_audit_df = pd.DataFrame(outlier_audit)
display(outlier_audit_df)

outlier_audit_df.to_csv(
    REPORT_DIR / "outlier_audit.csv",
    index=False
)


,file,column,valid_numeric_count,minimum,maximum,negative_count,iqr_lower_bound,iqr_upper_bound,statistical_outlier_count
0,orders.csv,order_total,160000,-78112.0,9999999.0,167,-9040.000,21088.000,12405
1,order_items.csv,quantity,314178,-6.0,500.0,307,-0.500,3.500,20379
2,order_items.csv,unit_price,315143,-41270.0,9999999.0,316,-2287.500,6220.500,26434
3,order_items.csv,discount_amount,283576,0.0,163062.0,0,-280.500,467.500,46288
4,products.csv,cost_price,3030,82.0,83205.0,0,-1514.125,4016.875,347
5,products.csv,weight_kg,2904,-1.0,629.0,10,-1.665,3.895,154
6,returns.csv,refund_amount,16048,0.0,250592.0,0,-3711.375,8697.625,1381
7,shipping.csv,shipping_fee,142083,-2577.0,9999999.0,159,-247.000,1193.000,9393
8,sellers.csv,rating,387,0.0,11.0,0,2.575,5.575,22


## 10. Broken Reference and Referential Integrity Audit

Check whether foreign-key values in orders, order items, payments, shipping and returns reference existing parent records. Missing or invalid relationships will be counted and documented before any records are repaired, mapped to an Unknown member, or rejected.

In [15]:

def read_processed(filename):
    return pd.read_csv(
        PROCESSED_DIR / filename,
        encoding="utf-8-sig",
        low_memory=False,
        dtype=str
    )

customers = read_processed("customers.csv")
products = read_processed("products.csv")
sellers = read_processed("sellers.csv")
locations = read_processed("locations.csv")
orders = read_processed("orders.csv")
items = read_processed("order_items.csv")
payments = read_processed("payments.csv")
shipping = read_processed("shipping.csv")
returns = read_processed("returns.csv")

reference_rules = [
    (orders, "customer_id", customers, "customer_id", "orders.customer_id"),
    (orders, "ship_city_id", locations, "city_id", "orders.ship_city_id"),
    (items, "order_id", orders, "order_id", "order_items.order_id"),
    (items, "product_id", products, "product_id", "order_items.product_id"),
    (items, "seller_id", sellers, "seller_id", "order_items.seller_id"),
    (payments, "order_id", orders, "order_id", "payments.order_id"),
    (shipping, "order_id", orders, "order_id", "shipping.order_id"),
]

reference_results = []

for child, child_col, parent, parent_col, label in reference_rules:
    if child_col not in child.columns or parent_col not in parent.columns:
        reference_results.append({
            "relationship": label,
            "status": "COLUMN NOT FOUND",
            "missing_reference_count": None
        })
        continue

    parent_ids = set(parent[parent_col].dropna().str.strip())
    child_ids = child[child_col].str.strip()

    missing_mask = child_ids.isna() | child_ids.eq("")
    invalid_mask = (
        ~missing_mask
        & ~child_ids.isin(parent_ids)
    )

    reference_results.append({
        "relationship": label,
        "status": "Audited",
        "missing_reference_count": int(missing_mask.sum()),
        "invalid_reference_count": int(invalid_mask.sum()),
        "rows_checked": len(child)
    })

reference_audit_df = pd.DataFrame(reference_results)
display(reference_audit_df)

reference_audit_df.to_csv(
    REPORT_DIR / "reference_integrity_audit.csv",
    index=False
)


,relationship,status,missing_reference_count,invalid_reference_count,rows_checked
0,orders.customer_id,Audited,0,321,160000
1,orders.ship_city_id,Audited,6388,153612,160000
2,order_items.order_id,Audited,0,0,315143
3,order_items.product_id,Audited,0,932,315143
4,order_items.seller_id,Audited,0,0,315143
5,payments.order_id,Audited,0,0,164815
6,shipping.order_id,Audited,0,0,151228


### 10.1 Investigate Key Formats and Reference Mismatches

Inspect the actual values and data types of parent and child keys for relationships with high invalid-reference counts. Confirm that the correct columns are being matched before deciding whether records contain broken references or require key normalization.

In [16]:

key_pairs = [
    (orders, "customer_id", customers, "customer_id",
     "orders → customers"),
    (orders, "ship_city_id", locations, "city_id",
     "orders → locations"),
    (items, "product_id", products, "product_id",
     "order_items → products"),
]

for child, child_col, parent, parent_col, label in key_pairs:
    print(f"\n--- {label} ---")
    print("Child key column:", child_col)
    print("Parent key column:", parent_col)

    print("\nChild examples:")
    display(
        child[child_col].dropna().astype(str).head(10).to_frame()
    )

    print("\nParent examples:")
    display(
        parent[parent_col].dropna().astype(str).head(10).to_frame()
    )

    child_keys = set(child[child_col].dropna().astype(str).str.strip())
    parent_keys = set(parent[parent_col].dropna().astype(str).str.strip())

    print("Unique child keys:", len(child_keys))
    print("Unique parent keys:", len(parent_keys))
    print("Example unmatched child keys:",
          list(child_keys - parent_keys)[:10])



--- orders → customers ---
Child key column: customer_id
Parent key column: customer_id

Child examples:


,customer_id
0,C042364
1,C022887
2,C010022
3,C032262
4,C035932
5,C023243
6,C017709
7,C002642
8,C033777
9,C012771



Parent examples:


,customer_id
0,C000001
1,C000002
2,C000003
3,C000004
4,C000005
5,C000006
6,C000007
7,C000008
8,C000009
9,C000010


Unique child keys: 33756
Unique parent keys: 45000
Example unmatched child keys: ['C96473', 'C91906', 'C98847', 'C98895', 'C96598', 'C92506', 'C95772', 'C96480', 'C90472', 'C96927']

--- orders → locations ---
Child key column: ship_city_id
Parent key column: city_id

Child examples:


...

Unique child keys: 650
Unique parent keys: 650
Example unmatched child keys: ['485.0', '380.0', '75.0', '107.0', '537.0', '467.0', '343.0', '342.0', '286.0', '257.0']

--- order_items → products ---
Child key column: product_id
Parent key column: product_id

Child examples:


,product_id
0,P02548
1,P01857
2,P01761
3,P00334
4,P02889
5,P03011
6,P00977
7,P01386
8,P00671
9,P03063



Parent examples:


,product_id
0,P00001
1,P00002
2,P00003
3,P00004
4,P00005
5,P00006
6,P00007
7,P00008
8,P00009
9,P00010


Unique child keys: 4086
Unique parent keys: 3200
Example unmatched child keys: ['P99713', 'P90959', 'P95591', 'P98672', 'P92372', 'P98286', 'P94852', 'P97280', 'P99877', 'P95570']


### 10.2 Test Safe Key Normalization

Test whether trimming whitespace and normalizing letter case improves reference matching. These are diagnostic checks only; they do not establish that two different identifiers represent the same entity.

In [17]:

for child, child_col, parent, parent_col, label in key_pairs:
    child_keys = child[child_col].dropna().astype(str).str.strip()
    parent_keys = parent[parent_col].dropna().astype(str).str.strip()

    exact_parent_keys = set(parent_keys)
    normalized_parent_keys = set(parent_keys.str.casefold())

    exact_matches = child_keys.isin(exact_parent_keys).sum()
    normalized_matches = (
        child_keys.str.casefold().isin(normalized_parent_keys).sum()
    )

    print(f"\n{label}")
    print("Child rows:", len(child))
    print("Exact matching rows:", int(exact_matches))
    print("Matching after case normalization:", int(normalized_matches))
    print("Additional matches from case normalization:",
          int(normalized_matches - exact_matches))



orders → customers
Child rows: 160000
Exact matching rows: 159679
Matching after case normalization: 159679
Additional matches from case normalization: 0

orders → locations
Child rows: 160000
Exact matching rows: 0
Matching after case normalization: 0
Additional matches from case normalization: 0

order_items → products
Child rows: 315143
Exact matching rows: 314211
Matching after case normalization: 314211
Additional matches from case normalization: 0


### 10.3 Normalize Location Keys and Recheck References

The shipping-city keys appear numeric, while the location keys are integer-like. Normalize only valid whole-number representations for comparison, then record unmatched values for investigation. Preserve the original identifiers in the source data.

In [18]:

# Work on copies; do not modify the original key columns.
orders_check = orders.copy()
locations_check = locations.copy()

orders_check["_city_key"] = pd.to_numeric(
    orders_check["ship_city_id"], errors="coerce"
)
locations_check["_city_key"] = pd.to_numeric(
    locations_check["city_id"], errors="coerce"
)

# Only accept whole numbers as valid location keys.
valid_location_keys = set(
    locations_check.loc[
        locations_check["_city_key"].notna()
        & (locations_check["_city_key"] % 1 == 0),
        "_city_key"
    ].astype("int64")
)

city_numeric = orders_check["_city_key"]
city_is_whole = city_numeric.notna() & (city_numeric % 1 == 0)

city_matched = (
    city_is_whole
    & city_numeric.fillna(-1).astype("int64").isin(valid_location_keys)
)

print("Orders checked:", len(orders_check))
print("Valid city references:", int(city_matched.sum()))
print("Missing city IDs:", int(orders_check["ship_city_id"].isna().sum()))
print("Non-missing invalid city references:",
      int((~city_matched & orders_check["ship_city_id"].notna()).sum()))

unmatched_cities = orders_check.loc[
    ~city_matched,
    ["order_id", "ship_city_id"]
].copy()

display(unmatched_cities.head(15))
unmatched_cities.to_csv(
    REPORT_DIR / "unmatched_order_city_references.csv",
    index=False
)


Orders checked: 160000
Valid city references: 153612
Missing city IDs: 6388
Non-missing invalid city references: 0


,order_id,ship_city_id
33,O0145255,NaN
36,O0029002,NaN
43,O0109710,NaN
47,O0064051,NaN
65,O0136515,NaN
122,O0089320,NaN
142,O0092044,NaN
149,O0069207,NaN
167,O0037442,NaN
175,O0069237,NaN


### 10.4 Document Unmatched Customer and Product References

Save unmatched customer and product keys with their related record identifiers. These records will not be silently deleted or remapped without evidence that identifies the correct master entity.

In [21]:

def save_unmatched(child, child_key, parent, parent_key, filename, label):
    child_values = child[child_key].astype("string").str.strip()
    parent_values = set(
        parent[parent_key].dropna().astype("string").str.strip()
    )

    unmatched_mask = (
        child_values.notna()
        & ~child_values.isin(parent_values)
    )

    unmatched = child.loc[unmatched_mask].copy()
    unmatched.to_csv(REPORT_DIR / filename, index=False)

    print(f"{label}: {len(unmatched)} unmatched rows")
    display(unmatched[[child_key]].drop_duplicates().head(10))
    return unmatched



unmatched_products = save_unmatched(
    items, "product_id", products, "product_id",
    "unmatched_order_products.csv",
    "Order items → products"
)



Order items → products: 932 unmatched rows


,product_id
127,P96243
167,P97524
234,P93742
554,P90073
561,P99790
618,P99751
749,P90470
991,P96552
1008,P95805
1388,P95644


### 10.5 Reference Audit Summary

Record the final counts for valid, missing, and unmatched references. These findings will guide documented repair, Unknown-member mapping, or quarantine decisions during warehouse loading.

In [22]:

reference_summary = pd.DataFrame([
    {
        "relationship": "orders → customers",
        "issue_count": len(unmatched_customers),
        "decision": "Investigate unmatched customer IDs"
    },
    {
        "relationship": "orders → locations",
        "issue_count": int(orders["ship_city_id"].isna().sum()),
        "decision": "Preserve missing city IDs; map to Unknown in warehouse"
    },
    {
        "relationship": "order_items → products",
        "issue_count": len(unmatched_products),
        "decision": "Investigate unmatched product IDs"
    }
])

display(reference_summary)

reference_summary.to_csv(
    REPORT_DIR / "reference_audit_summary.csv",
    index=False
)


,relationship,issue_count,decision
0,orders → customers,321,Investigate unmatched customer IDs
1,orders → locations,6388,Preserve missing city IDs; map to Unknown in w...
2,order_items → products,932,Investigate unmatched product IDs


## 11. Standardize Text and Categories

Normalize whitespace and case in categorical fields, and standardize known status and channel variants. Preserve identifier values and descriptive names. Missing values are not replaced with invented information, and every transformation is recorded for review.

In [23]:

from pathlib import Path
import pandas as pd
import re

category_columns = {
    "orders.csv": [
        "order_status", "channel", "discount_code"
    ],
    "payments.csv": [
        "payment_method", "payment_status"
    ],
    "products.csv": [
        "category", "sub_category"
    ],
    "returns.csv": [
        "return_status", "return_reason"
    ],
    "sellers.csv": ["seller_tier"],
    "shipping.csv": ["courier", "delivery_status"],
    "customers.csv": ["gender"],
}

# Canonical labels for known spelling/case variants.
canonical_values = {
    "order_status": {
        "deliverd": "Delivered",
        "delivered": "Delivered",
        "complete": "Completed",
        "completed": "Completed",
        "cancelled": "Cancelled",
        "canceled": "Cancelled",
        "in transit": "In Transit",
        "returned": "Returned",
    },
    "payment_method": {
        "cod": "COD",
    },
    "payment_status": {
        "paid": "Paid",
        "pending": "Pending",
        "failed": "Failed",
    },
    "delivery_status": {
        "delivered": "Delivered",
        "in transit": "In Transit",
        "returned to seller": "Returned to Seller",
    },
}

text_cleaning_log = []

for filename, columns in category_columns.items():
    path = PROCESSED_DIR / filename
    if not path.exists():
        print("Skipped missing file:", filename)
        continue

    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False,
                     dtype=str, keep_default_na=True)

    # Remove leading/trailing whitespace and repeated internal spaces
    # from text fields, without changing IDs or people's names.
    for col in df.select_dtypes(include="object").columns:
        before = df[col].copy()
        cleaned = df[col].str.strip().str.replace(r"\s+", " ", regex=True)
        changed = int((before.fillna("") != cleaned.fillna("")).sum())
        if changed:
            text_cleaning_log.append({
                "file": filename, "column": col,
                "action": "Trim/normalize whitespace",
                "rows_changed": changed
            })
        df[col] = cleaned

    for col in columns:
        if col not in df.columns:
            continue

        before = df[col].copy()
        normalized = df[col].str.strip().str.casefold()

        # Standardize recognized categories; title-case other non-empty
        # category values, but do not invent values for missing entries.
        mapping = canonical_values.get(col, {})
        standardized = normalized.map(mapping)
        fallback = normalized.str.title()
        df[col] = standardized.fillna(fallback).where(
            normalized.notna(), df[col]
        )

        changed = int((before.fillna("") != df[col].fillna("")).sum())
        if changed:
            text_cleaning_log.append({
                "file": filename, "column": col,
                "action": "Standardize category labels",
                "rows_changed": changed
            })

    df.to_csv(path, index=False)

text_cleaning_log_df = pd.DataFrame(text_cleaning_log)
display(text_cleaning_log_df.head(30))
text_cleaning_log_df.to_csv(
    REPORT_DIR / "text_standardization_log.csv", index=False
)
print("Text standardization log saved.")


,file,column,action,rows_changed
0,orders.csv,order_status,Trim/normalize whitespace,903
1,orders.csv,discount_code,Trim/normalize whitespace,2057
2,orders.csv,order_status,Standardize category labels,31368
3,orders.csv,channel,Standardize category labels,17444
4,orders.csv,discount_code,Standardize category labels,48139
5,payments.csv,payment_method,Trim/normalize whitespace,5263
6,payments.csv,payment_status,Trim/normalize whitespace,742
7,payments.csv,payment_method,Standardize category labels,49726
8,payments.csv,payment_status,Standardize category labels,21126
9,products.csv,product_name,Trim/normalize whitespace,123


Text standardization log saved.


## 12. Numeric Type Conversion and Invalid-Value Logging

Convert numeric fields to numeric types where possible. Recognized number words and currency formatting are normalized using explicit rules. Values that remain unparseable are logged with their source row and original value; suspicious numeric values are flagged for investigation rather than automatically deleted or corrected.

In [24]:

numeric_columns = {
    "orders.csv": ["order_total"],
    "order_items.csv": [
        "quantity", "unit_price", "discount_amount"
    ],
    "payments.csv": ["amount"],
    "products.csv": ["cost_price", "weight_kg"],
    "returns.csv": ["refund_amount"],
    "shipping.csv": ["shipping_fee"],
    "sellers.csv": ["rating"],
    "locations.csv": ["latitude", "longitude"],
}

number_words = {
    "zero": 0, "one": 1, "two": 2, "three": 3, "four": 4,
    "five": 5, "six": 6, "seven": 7, "eight": 8, "nine": 9,
    "ten": 10,
}

numeric_conversion_log = []
unparseable_numeric_records = []

for filename, columns in numeric_columns.items():
    path = PROCESSED_DIR / filename
    if not path.exists():
        continue

    df = pd.read_csv(
        path, encoding="utf-8-sig", low_memory=False, dtype=str
    )

    for col in columns:
        if col not in df.columns:
            continue

        raw = df[col].copy()
        cleaned = raw.str.strip().str.casefold()

        # Convert simple English number words for quantity.
        if col == "quantity":
            cleaned = cleaned.replace(number_words)

        # Remove currency labels and thousands separators.
        cleaned = cleaned.str.replace(r"(?i)\brs\.?\s*", "", regex=True)
        cleaned = cleaned.str.replace(",", "", regex=False)
        cleaned = cleaned.replace({"": pd.NA, "nan": pd.NA, "none": pd.NA})

        converted = pd.to_numeric(cleaned, errors="coerce")

        bad_mask = raw.notna() & raw.str.strip().ne("") & converted.isna()

        if bad_mask.any():
            bad = pd.DataFrame({
                "file": filename,
                "source_row_number": raw.index[bad_mask] + 2,
                "column": col,
                "original_value": raw[bad_mask].values,
                "reason": "UNPARSEABLE_NUMERIC"
            })
            unparseable_numeric_records.append(bad)

        numeric_conversion_log.append({
            "file": filename,
            "column": col,
            "non_missing_before": int(raw.notna().sum()),
            "unparseable_non_empty": int(bad_mask.sum()),
            "converted_non_missing": int(converted.notna().sum())
        })

        # Store the converted values; unresolved values become blank,
        # with their original value preserved in the reject/audit report.
        df[col] = converted

    df.to_csv(path, index=False)

numeric_log_df = pd.DataFrame(numeric_conversion_log)
display(numeric_log_df)

numeric_log_df.to_csv(
    REPORT_DIR / "numeric_conversion_log.csv", index=False
)

if unparseable_numeric_records:
    unparseable_numeric_df = pd.concat(
        unparseable_numeric_records, ignore_index=True
    )
else:
    unparseable_numeric_df = pd.DataFrame(
        columns=["file", "source_row_number", "column",
                 "original_value", "reason"]
    )

display(unparseable_numeric_df.head(20))
unparseable_numeric_df.to_csv(
    REPORT_DIR / "unparseable_numeric_values.csv", index=False
)

print("Numeric conversion complete; unresolved values logged.")


,file,column,non_missing_before,unparseable_non_empty,converted_non_missing
0,orders.csv,order_total,160000,0,160000
1,order_items.csv,quantity,315143,965,314178
2,order_items.csv,unit_price,315143,0,315143
3,order_items.csv,discount_amount,283576,0,283576
4,payments.csv,amount,164815,0,164815
5,products.csv,cost_price,3030,0,3030
6,products.csv,weight_kg,2904,0,2904
7,returns.csv,refund_amount,16048,0,16048
8,shipping.csv,shipping_fee,142083,0,142083
9,sellers.csv,rating,387,0,387


,file,source_row_number,column,original_value,reason
0,order_items.csv,235,quantity,one,UNPARSEABLE_NUMERIC
1,order_items.csv,624,quantity,one,UNPARSEABLE_NUMERIC
2,order_items.csv,750,quantity,three,UNPARSEABLE_NUMERIC
3,order_items.csv,1097,quantity,four,UNPARSEABLE_NUMERIC
4,order_items.csv,1719,quantity,one,UNPARSEABLE_NUMERIC
5,order_items.csv,2230,quantity,one,UNPARSEABLE_NUMERIC
6,order_items.csv,2616,quantity,one,UNPARSEABLE_NUMERIC
7,order_items.csv,2643,quantity,one,UNPARSEABLE_NUMERIC
8,order_items.csv,2661,quantity,one,UNPARSEABLE_NUMERIC
9,order_items.csv,3501,quantity,two,UNPARSEABLE_NUMERIC


Numeric conversion complete; unresolved values logged.


## 13. Apply Approved Missing-Value Rules

Apply only missing-value rules supported by the challenge brief. An empty discount code means no discount; missing descriptive categories are labelled Unknown where appropriate. Other missing values are retained until their business meaning is established.

In [25]:

missing_rule_log = []

# Empty discount code explicitly means no discount in the brief.
orders_path = PROCESSED_DIR / "orders.csv"
orders_df = pd.read_csv(orders_path, encoding="utf-8-sig",
                        low_memory=False, dtype=str)

if "discount_code" in orders_df.columns:
    missing_before = int(
        orders_df["discount_code"].isna().sum()
        + orders_df["discount_code"].fillna("").str.strip().eq("").sum()
    )
    orders_df["discount_code"] = (
        orders_df["discount_code"].fillna("").str.strip()
    )
    orders_df.loc[
        orders_df["discount_code"].eq(""), "discount_code"
    ] = "NO_DISCOUNT"

    missing_rule_log.append({
        "file": "orders.csv",
        "column": "discount_code",
        "rule": "Blank means no discount",
        "rows_affected": missing_before
    })

orders_df.to_csv(orders_path, index=False)

# These are descriptive labels, not invented factual values.
unknown_category_rules = {
    "customers.csv": ["gender"],
    "products.csv": ["sub_category", "brand"],
    "returns.csv": ["return_reason"],
    "sellers.csv": ["seller_tier"],
}

for filename, columns in unknown_category_rules.items():
    path = PROCESSED_DIR / filename
    df = pd.read_csv(path, encoding="utf-8-sig",
                     low_memory=False, dtype=str)

    for col in columns:
        if col not in df.columns:
            continue

        missing_mask = df[col].isna() | df[col].str.strip().eq("")
        affected = int(missing_mask.sum())
        df.loc[missing_mask, col] = "Unknown"

        missing_rule_log.append({
            "file": filename,
            "column": col,
            "rule": "Label missing descriptive category as Unknown",
            "rows_affected": affected
        })

    df.to_csv(path, index=False)

missing_rule_log_df = pd.DataFrame(missing_rule_log)
display(missing_rule_log_df)
missing_rule_log_df.to_csv(
    REPORT_DIR / "approved_missing_value_actions.csv",
    index=False
)


,file,column,rule,rows_affected
0,orders.csv,discount_code,Blank means no discount,219742
1,customers.csv,gender,Label missing descriptive category as Unknown,2199
2,products.csv,sub_category,Label missing descriptive category as Unknown,485
3,products.csv,brand,Label missing descriptive category as Unknown,392
4,returns.csv,return_reason,Label missing descriptive category as Unknown,2390
5,sellers.csv,seller_tier,Label missing descriptive category as Unknown,22


## 14. Outlier and Business-Rule Validation

Flag negative monetary values, non-positive quantities and unit prices, and non-positive product weights for investigation. Statistical outliers are retained because extreme values may be legitimate business transactions. No suspicious value is automatically deleted or capped.

In [26]:

validation_rules = {
    "orders.csv": {"order_total": "negative"},
    "order_items.csv": {
        "quantity": "non_positive",
        "unit_price": "non_positive",
        "discount_amount": "negative",
    },
    "payments.csv": {"amount": "negative"},
    "products.csv": {
        "cost_price": "negative",
        "weight_kg": "non_positive",
    },
    "returns.csv": {"refund_amount": "negative"},
    "shipping.csv": {"shipping_fee": "negative"},
}

business_rule_issues = []

for filename, columns in validation_rules.items():
    path = PROCESSED_DIR / filename
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)

    for col, rule in columns.items():
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col], errors="coerce")

        if rule == "negative":
            mask = values < 0
        else:
            mask = values <= 0

        if mask.any():
            flagged = df.loc[mask].copy()
            flagged.insert(0, "source_file", filename)
            flagged.insert(1, "source_row_number", flagged.index + 2)
            flagged.insert(2, "flagged_column", col)
            flagged.insert(3, "flag_reason", rule)
            business_rule_issues.append(flagged)

if business_rule_issues:
    business_rule_issues_df = pd.concat(
        business_rule_issues, ignore_index=True, sort=False
    )
else:
    business_rule_issues_df = pd.DataFrame()

display(business_rule_issues_df.head(25))
business_rule_issues_df.to_csv(
    REPORT_DIR / "business_rule_issues.csv", index=False
)

print("Flagged rows:", len(business_rule_issues_df))
print("Report saved:", REPORT_DIR / "business_rule_issues.csv")


,source_file,source_row_number,flagged_column,flag_reason,order_id,customer_id,order_date,order_status,ship_city_id,order_total,...,brand,list_price,cost_price,weight_kg,shipment_id,courier,shipped_date,delivered_date,shipping_fee,delivery_status
0,orders.csv,330,order_total,negative,O0114255,C007109,2025-02-28,Delivered,319.0,-21979.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,orders.csv,2564,order_total,negative,O0034353,C000951,2024-06-01,Completed,434.0,-8680.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,orders.csv,2679,order_total,negative,O0046326,C001588,2024-07-14,Delivered,450.0,-3445.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,orders.csv,3139,order_total,negative,O0022010,C036413,2024-04-08,Delivered,347.0,-3467.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,orders.csv,3429,order_total,negative,O0147331,C031284,2025-05-31,Delivered,450.0,-4643.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,orders.csv,5144,order_total,negative,O0041493,C004062,24/06/2024,Delivered,281.0,-8501.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,orders.csv,6158,order_total,negative,O0141234,C034178,2025-05-13,Delivered,614.0,-6485.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,orders.csv,9356,order_total,negative,O0055384,C018831,2024-08-17,Delivered,597.0,-3526.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,orders.csv,10508,order_total,negative,O0155078,C039641,2025-06-16,Cancelled,255.0,-1128.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,orders.csv,11478,order_total,negative,O0042398,C007578,2024-06-28,Delivered,614.0,-23268.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Flagged rows: 1953
Report saved: c:\Users\LENOVO\Documents\CodeConnect26_DataAnalytics\reports\business_rule_issues.csv


## 15. Cleaning Checkpoint and Row-Count Reconciliation

Record the current row counts, remaining exact duplicates, and missing values in the processed datasets. This checkpoint distinguishes completed transformations from issues that still require investigation before warehouse loading.

In [27]:

checkpoint = []

for path in sorted(PROCESSED_DIR.glob("*.csv")):
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)

    checkpoint.append({
        "file": path.name,
        "rows": len(df),
        "columns": len(df.columns),
        "exact_duplicates_remaining": int(df.duplicated().sum()),
        "missing_cells_remaining": int(df.isna().sum().sum()),
    })

checkpoint_df = pd.DataFrame(checkpoint)
display(checkpoint_df)

checkpoint_df.to_csv(
    REPORT_DIR / "cleaning_checkpoint.csv", index=False
)

print("Checkpoint saved.")
print("Total processed rows:", int(checkpoint_df["rows"].sum()))
print(
    "Exact duplicates remain:",
    int(checkpoint_df["exact_duplicates_remaining"].sum())
)


,file,rows,columns,exact_duplicates_remaining,missing_cells_remaining
0,customers.csv,45000,8,0,12130
1,locations.csv,650,7,0,89
2,order_items.csv,315143,7,0,32532
3,orders.csv,160000,8,0,6388
4,payments.csv,164815,6,0,4927
5,products.csv,3200,9,0,466
6,returns.csv,16048,6,0,0
7,sellers.csv,420,6,0,67
8,shipping.csv,151228,7,0,33929


Checkpoint saved.
Total processed rows: 856504
Exact duplicates remain: 0


## 16. Date Repair and Quarantine

Parse dates using documented formats, including day/month/year for slash-separated dates. Repair dates only when related records provide sufficient evidence. Preserve unrepairable or out-of-range records in a reject file with a reason code, and log before-and-after counts.

In [28]:

from datetime import datetime

def parse_nexus_date(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NaT

    value = str(value).strip()

    formats = [
        "%Y-%m-%d",
        "%d/%m/%Y",   # Challenge specifies day/month/year
        "%d-%m-%Y",
        "%b %d %Y",
        "%B %d %Y",
        "%Y-%m-%d %H:%M:%S",
    ]

    for fmt in formats:
        try:
            return pd.Timestamp(datetime.strptime(value, fmt))
        except ValueError:
            continue

    return pd.NaT


date_review = []

date_columns = {
    "orders.csv": ["order_date"],
    "payments.csv": ["payment_date"],
    "shipping.csv": ["shipped_date", "delivered_date"],
    "returns.csv": ["return_date"],
    "sellers.csv": ["join_date"],
}

for filename, columns in date_columns.items():
    path = PROCESSED_DIR / filename
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig",
                     low_memory=False)

    for col in columns:
        if col not in df.columns:
            continue

        parsed = df[col].map(parse_nexus_date)
        invalid = df[col].notna() & df[col].str.strip().ne("") & parsed.isna()

        if filename == "orders.csv":
            out_of_range = parsed.notna() & (
                (parsed < pd.Timestamp("2024-01-01"))
                | (parsed > pd.Timestamp("2025-06-30"))
            )
        else:
            out_of_range = pd.Series(False, index=df.index)

        issue_mask = invalid | out_of_range

        for idx in df.index[issue_mask]:
            date_review.append({
                "file": filename,
                "source_row_number": idx + 2,
                "record_id": (
                    df.loc[idx, "order_id"]
                    if "order_id" in df.columns else
                    df.loc[idx, "payment_id"]
                    if "payment_id" in df.columns else
                    df.loc[idx, "shipment_id"]
                    if "shipment_id" in df.columns else
                    df.loc[idx, "return_id"]
                    if "return_id" in df.columns else
                    df.loc[idx, "seller_id"]
                    if "seller_id" in df.columns else ""
                ),
                "column": col,
                "original_value": df.loc[idx, col],
                "parsed_candidate": (
                    parsed.loc[idx].strftime("%Y-%m-%d")
                    if pd.notna(parsed.loc[idx]) else ""
                ),
                "reason_code": (
                    "INVALID_DATE" if invalid.loc[idx]
                    else "OUT_OF_ALLOWED_ORDER_PERIOD"
                ),
                "action": "REVIEW_BEFORE_REPAIR_OR_QUARANTINE"
            })

date_review_df = pd.DataFrame(date_review)
display(date_review_df.groupby(
    ["file", "column", "reason_code"]
).size().reset_index(name="records"))

date_review_df.to_csv(
    REPORT_DIR / "date_repair_review.csv", index=False
)
print("Date review saved; source data unchanged.")


,file,column,reason_code,records
0,orders.csv,order_date,INVALID_DATE,1124
1,orders.csv,order_date,OUT_OF_ALLOWED_ORDER_PERIOD,481
2,payments.csv,payment_date,INVALID_DATE,1067
3,returns.csv,return_date,INVALID_DATE,136
4,sellers.csv,join_date,INVALID_DATE,4
5,shipping.csv,delivered_date,INVALID_DATE,924
6,shipping.csv,shipped_date,INVALID_DATE,1022


Date review saved; source data unchanged.


### 16.1 Quarantine Unrepairable Date Records

Retain records with invalid non-empty dates or out-of-period order dates in a separate reject file with a reason code. Do not guess a replacement date when supporting evidence is insufficient. Keep valid records in the processed datasets and record before-and-after counts.

In [29]:

from datetime import datetime

def parse_nexus_date(value):
    if pd.isna(value) or not str(value).strip():
        return pd.NaT

    value = str(value).strip()
    formats = [
        "%Y-%m-%d",
        "%d/%m/%Y",
        "%d-%m-%Y",
        "%b %d %Y",
        "%B %d %Y",
        "%Y-%m-%d %H:%M:%S",
    ]

    for fmt in formats:
        try:
            return pd.Timestamp(datetime.strptime(value, fmt))
        except ValueError:
            pass

    return pd.NaT


date_columns = {
    "orders.csv": ["order_date"],
    "payments.csv": ["payment_date"],
    "shipping.csv": ["shipped_date", "delivered_date"],
    "returns.csv": ["return_date"],
    "sellers.csv": ["join_date"],
}

date_rejects = []
date_quarantine_log = []

for filename, columns in date_columns.items():
    path = PROCESSED_DIR / filename
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig",
                     low_memory=False)

    row_reasons = {}
    parsed_columns = {}

    for col in columns:
        if col not in df.columns:
            continue

        raw = df[col]
        parsed = raw.map(parse_nexus_date)
        parsed_columns[col] = parsed

        nonempty = raw.notna() & raw.str.strip().ne("")
        invalid = nonempty & parsed.isna()

        if filename == "orders.csv" and col == "order_date":
            out_of_range = parsed.notna() & (
                (parsed < pd.Timestamp("2024-01-01"))
                | (parsed > pd.Timestamp("2025-06-30"))
            )
        else:
            out_of_range = pd.Series(False, index=df.index)

        for idx in df.index[invalid]:
            row_reasons.setdefault(idx, []).append(
                f"{col}:INVALID_DATE"
            )

        for idx in df.index[out_of_range]:
            row_reasons.setdefault(idx, []).append(
                f"{col}:OUT_OF_ALLOWED_ORDER_PERIOD"
            )

    reject_indices = sorted(row_reasons.keys())

    if reject_indices:
        rejected = df.loc[reject_indices].copy()
        rejected.insert(0, "source_file", filename)
        rejected.insert(1, "source_row_number",
                        [idx + 2 for idx in reject_indices])
        rejected.insert(
            2, "reason_code",
            [";".join(row_reasons[idx]) for idx in reject_indices]
        )
        date_rejects.append(rejected)

    before_count = len(df)
    df_clean = df.drop(index=reject_indices).copy()

    # Normalize remaining valid date strings to ISO YYYY-MM-DD.
    for col, parsed in parsed_columns.items():
        df_clean[col] = parsed.loc[df_clean.index].map(
            lambda x: x.strftime("%Y-%m-%d") if pd.notna(x) else ""
        )

    df_clean.to_csv(path, index=False)

    date_quarantine_log.append({
        "file": filename,
        "rows_before": before_count,
        "rows_quarantined": len(reject_indices),
        "rows_after": len(df_clean),
        "reconciles": before_count == len(df_clean) + len(reject_indices)
    })

if date_rejects:
    date_rejects_df = pd.concat(date_rejects, ignore_index=True,
                                sort=False)
else:
    date_rejects_df = pd.DataFrame()

date_rejects_df.to_csv(
    REJECT_DIR / "date_rejects.csv", index=False
)

date_quarantine_log_df = pd.DataFrame(date_quarantine_log)
display(date_quarantine_log_df)

date_quarantine_log_df.to_csv(
    REPORT_DIR / "date_quarantine_log.csv", index=False
)

print("Rejected date rows:", len(date_rejects_df))
print("Date rejects saved:", REJECT_DIR / "date_rejects.csv")
print("All row counts reconcile:",
      date_quarantine_log_df["reconciles"].all())


,file,rows_before,rows_quarantined,rows_after,reconciles
0,orders.csv,160000,1605,158395,True
1,payments.csv,164815,1067,163748,True
2,shipping.csv,151228,1942,149286,True
3,returns.csv,16048,136,15912,True
4,sellers.csv,420,4,416,True


Rejected date rows: 4754
Date rejects saved: c:\Users\LENOVO\Documents\CodeConnect26_DataAnalytics\data\rejected\date_rejects.csv
All row counts reconcile: True


## 17. Duplicate Entity Investigation

Identify potential duplicate customers and products using reliable identifiers and supporting attributes. Names alone are insufficient evidence for merging. Candidate matches will be saved for review before any master-ID mapping is applied.

In [30]:

import pandas as pd
from pathlib import Path

def load_processed(filename):
    return pd.read_csv(
        PROCESSED_DIR / filename,
        dtype=str,
        encoding="utf-8-sig",
        low_memory=False
    ).fillna("")

def norm(series):
    return series.astype(str).str.strip().str.casefold()

customers = load_processed("customers.csv")
products = load_processed("products.csv")

def find_duplicate_candidates(df, entity, id_col, rules):
    results = []

    for rule_name, columns in rules:
        if id_col not in df.columns or not all(c in df.columns for c in columns):
            continue

        temp = df[[id_col] + columns].copy()

        # Only use records where every field in this matching rule exists.
        for col in columns:
            temp[col] = norm(temp[col])

        valid = temp[columns].ne("").all(axis=1)
        temp = temp.loc[valid]

        for _, group in temp.groupby(columns, dropna=False):
            ids = sorted(group[id_col].astype(str).unique())

            if len(ids) > 1:
                results.append({
                    "entity": entity,
                    "matching_rule": rule_name,
                    "matching_columns": ", ".join(columns),
                    "candidate_ids": " | ".join(ids),
                    "candidate_count": len(ids),
                    "review_status": "Needs manual evidence review"
                })

    return results

customer_rules = [
    ("Exact email", ["email"]),
    ("Exact phone", ["phone"]),
    ("Name + email", ["full_name", "email"]),
    ("Name + phone", ["full_name", "phone"]),
]

product_rules = [
    ("Name + brand", ["product_name", "brand"]),
    ("Name + category + brand", ["product_name", "category", "brand"]),
    ("Name + subcategory + brand", ["product_name", "sub_category", "brand"]),
]

customer_candidates = find_duplicate_candidates(
    customers, "customer", "customer_id", customer_rules
)
product_candidates = find_duplicate_candidates(
    products, "product", "product_id", product_rules
)

candidate_df = pd.DataFrame(customer_candidates + product_candidates)

candidate_path = REPORT_DIR / "duplicate_entity_candidates.csv"
candidate_df.to_csv(candidate_path, index=False)

print("Customer candidate groups:", len(customer_candidates))
print("Product candidate groups:", len(product_candidates))
print("Candidate report:", candidate_path)
display(candidate_df.head(20))


Customer candidate groups: 5012
Product candidate groups: 43
Candidate report: c:\Users\LENOVO\Documents\CodeConnect26_DataAnalytics\reports\duplicate_entity_candidates.csv


,entity,matching_rule,matching_columns,candidate_ids,candidate_count,review_status
0,customer,Exact email,email,C004431 | C042043,2,Needs manual evidence review
1,customer,Exact email,email,C041580 | C044707,2,Needs manual evidence review
2,customer,Exact email,email,C040741 | C043556,2,Needs manual evidence review
3,customer,Exact email,email,C028914 | C042681,2,Needs manual evidence review
4,customer,Exact email,email,C019252 | C044977,2,Needs manual evidence review
5,customer,Exact email,email,C014284 | C042504,2,Needs manual evidence review
6,customer,Exact email,email,C039005 | C044755,2,Needs manual evidence review
7,customer,Exact email,email,C028567 | C043722,2,Needs manual evidence review
8,customer,Exact email,email,C030067 | C044904,2,Needs manual evidence review
9,customer,Exact email,email,C018944 | C042669,2,Needs manual evidence review


## 18. Numeric Outlier Review

Review implausible monetary values, quantities, weights, and ratings. Statistical outliers are investigation candidates, not automatic errors. Correct values only when supported by the source data or business rules; otherwise preserve them with a documented flag or quarantine reason.

In [31]:

numeric_review = []

numeric_targets = {
    "orders.csv": ["order_total"],
    "order_items.csv": ["quantity", "unit_price", "discount_amount"],
    "payments.csv": ["amount"],
    "products.csv": ["cost_price", "weight_kg"],
    "returns.csv": ["refund_amount"],
    "shipping.csv": ["shipping_fee"],
    "sellers.csv": ["rating"],
}

for filename, columns in numeric_targets.items():
    df = load_processed(filename)

    for col in columns:
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col].str.strip(), errors="coerce")
        nonempty = df[col].str.strip().ne("")
        invalid_numeric = nonempty & values.isna()

        numeric = values.dropna()
        numeric_review.append({
            "file": filename,
            "column": col,
            "nonempty_values": int(nonempty.sum()),
            "unparseable_values": int(invalid_numeric.sum()),
            "negative_values": int((numeric < 0).sum()),
            "minimum": numeric.min() if len(numeric) else None,
            "maximum": numeric.max() if len(numeric) else None,
        })

numeric_review_df = pd.DataFrame(numeric_review)
display(numeric_review_df)

numeric_review_df.to_csv(
    REPORT_DIR / "numeric_review.csv", index=False
)


,file,column,nonempty_values,unparseable_values,negative_values,minimum,maximum
0,orders.csv,order_total,158395,0,163,-78112.0,9999999.0
1,order_items.csv,quantity,314178,0,307,-6.0,500.0
2,order_items.csv,unit_price,315143,0,316,-41270.0,9999999.0
3,order_items.csv,discount_amount,283576,0,0,0.0,163062.0
4,payments.csv,amount,163748,0,656,-143168.0,9999999.0
5,products.csv,cost_price,3030,0,0,82.0,83205.0
6,products.csv,weight_kg,2904,0,10,-1.0,629.0
7,returns.csv,refund_amount,15912,0,0,0.0,250592.0
8,shipping.csv,shipping_fee,140253,0,155,-2577.0,9999999.0
9,sellers.csv,rating,384,0,0,0.0,11.0


## 19. Standardize Categorical Labels

Trim unnecessary whitespace and standardize casing for categorical fields such as order status, payment method/status, product categories, seller tier, courier, and delivery status. This normalization does not merge different business meanings or invent missing values.

In [32]:

category_columns = {
    "orders.csv": ["order_status", "discount_code"],
    "payments.csv": ["payment_method", "payment_status"],
    "products.csv": ["category", "sub_category", "brand"],
    "sellers.csv": ["seller_tier"],
    "shipping.csv": ["courier", "delivery_status"],
    "returns.csv": ["return_status", "return_reason"],
}

category_log = []

for filename, columns in category_columns.items():
    path = PROCESSED_DIR / filename
    df = pd.read_csv(path, dtype=str, encoding="utf-8-sig",
                     low_memory=False).fillna("")

    for col in columns:
        if col not in df.columns:
            continue

        before = df[col].copy()
        # Case/spacing normalization only; preserve blank values.
        df[col] = before.str.strip()
        nonblank = df[col].ne("")
        df.loc[nonblank, col] = df.loc[nonblank, col].str.title()

        category_log.append({
            "file": filename,
            "column": col,
            "nonblank_values": int(nonblank.sum()),
            "values_changed": int((before != df[col]).sum()),
            "distinct_values_after": int(df.loc[nonblank, col].nunique())
        })

    # Empty discount codes mean no discount per challenge brief.
    if filename == "orders.csv" and "discount_code" in df.columns:
        df["discount_code"] = df["discount_code"].str.strip()

    df.to_csv(path, index=False)

category_log_df = pd.DataFrame(category_log)
display(category_log_df)

category_log_df.to_csv(
    REPORT_DIR / "category_standardization_log.csv", index=False
)
print("Category normalization log saved.")


,file,column,nonblank_values,values_changed,distinct_values_after
0,orders.csv,order_status,158395,0,8
1,orders.csv,discount_code,158395,108765,9
2,payments.csv,payment_method,163748,73548,11
3,payments.csv,payment_status,163748,0,8
4,products.csv,category,3200,0,38
5,products.csv,sub_category,3200,0,37
6,products.csv,brand,3200,1147,17
7,sellers.csv,seller_tier,416,0,4
8,shipping.csv,courier,149286,0,13
9,shipping.csv,delivery_status,149286,4879,10


Category normalization log saved.


## 20. Review Distinct Category Values

Review the normalized values in categorical columns to identify spelling variants, inconsistent labels, and unexpected categories. Only equivalent labels will be mapped to a common canonical value; distinct business meanings will be preserved.

In [33]:

category_review = []

for filename, columns in category_columns.items():
    df = load_processed(filename)

    for col in columns:
        if col not in df.columns:
            continue

        counts = df[col].astype(str).str.strip()
        counts = counts[counts.ne("")].value_counts()

        for value, count in counts.items():
            category_review.append({
                "file": filename,
                "column": col,
                "value": value,
                "count": int(count)
            })

category_review_df = pd.DataFrame(category_review)

display(
    category_review_df.sort_values(
        ["file", "column", "value"]
    ).reset_index(drop=True)
)

category_review_df.to_csv(
    REPORT_DIR / "distinct_category_values.csv",
    index=False
)

print("Distinct category report saved.")


,file,column,value,count
0,orders.csv,discount_code,No_Discount,108765
1,orders.csv,discount_code,Save10,8118
2,orders.csv,discount_code,Save15,8609
3,orders.csv,discount_code,Save20,7494
4,orders.csv,discount_code,Save25,4553
...,...,...,...,...
189,shipping.csv,delivery_status,In-Transit,76
190,shipping.csv,delivery_status,Return To Seller,492
191,shipping.csv,delivery_status,Returned To Seller,4879
192,shipping.csv,delivery_status,Rto,491


Distinct category report saved.


## 21. Investigate Implausible Numeric Values

Identify records with invalid negative values or suspiciously extreme numeric values. Statistical outliers are not automatically errors. Records will be corrected only when reliable evidence supports the correction; otherwise, they will be flagged or quarantined with a reason code and preserved for review.

In [34]:

numeric_rules = {
    "orders.csv": {
        "order_total": {"negative": True, "extreme": 1000000}
    },
    "order_items.csv": {
        "quantity": {"negative": True, "extreme": 100},
        "unit_price": {"negative": True, "extreme": 1000000},
        "discount_amount": {"negative": False, "extreme": 1000000}
    },
    "payments.csv": {
        "amount": {"negative": True, "extreme": 1000000}
    },
    "products.csv": {
        "cost_price": {"negative": True, "extreme": 1000000},
        "weight_kg": {"negative": True, "extreme": 100}
    },
    "returns.csv": {
        "refund_amount": {"negative": True, "extreme": 1000000}
    },
    "shipping.csv": {
        "shipping_fee": {"negative": True, "extreme": 1000000}
    },
    "sellers.csv": {
        "rating": {"negative": True, "extreme": 5}
    }
}

numeric_issues = []

for filename, rules in numeric_rules.items():
    df = load_processed(filename)

    for col, rule in rules.items():
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col], errors="coerce")
        issue = pd.Series("", index=df.index, dtype="object")

        if rule["negative"]:
            issue.loc[values < 0] = "NEGATIVE_VALUE"

        # These are investigation thresholds, not automatic corrections.
        extreme_mask = values.abs() > rule["extreme"]
        issue.loc[extreme_mask & issue.eq("")] = "EXTREME_VALUE"

        affected = df.loc[issue.ne("")].copy()

        if not affected.empty:
            affected["source_file"] = filename
            affected["source_row_index"] = affected.index
            affected["issue_column"] = col
            affected["observed_value"] = values.loc[affected.index]
            affected["reason_code"] = issue.loc[affected.index].values

            numeric_issues.append(affected)

if numeric_issues:
    numeric_issue_df = pd.concat(numeric_issues, ignore_index=True)
else:
    numeric_issue_df = pd.DataFrame(
        columns=["source_file", "source_row_index",
                 "issue_column", "observed_value", "reason_code"]
    )

display(
    numeric_issue_df[
        ["source_file", "source_row_index", "issue_column",
         "observed_value", "reason_code"]
    ].groupby(
        ["source_file", "issue_column", "reason_code"],
        dropna=False
    ).size().reset_index(name="records")
)

numeric_issue_df.to_csv(
    REPORT_DIR / "numeric_records_for_review.csv",
    index=False
)

print("Numeric review records:", len(numeric_issue_df))
print("Saved to reports/numeric_records_for_review.csv")


,source_file,issue_column,reason_code,records
0,order_items.csv,quantity,EXTREME_VALUE,641
1,order_items.csv,quantity,NEGATIVE_VALUE,307
2,order_items.csv,unit_price,EXTREME_VALUE,318
3,order_items.csv,unit_price,NEGATIVE_VALUE,316
4,orders.csv,order_total,EXTREME_VALUE,145
5,orders.csv,order_total,NEGATIVE_VALUE,163
6,payments.csv,amount,EXTREME_VALUE,162
7,payments.csv,amount,NEGATIVE_VALUE,656
8,products.csv,weight_kg,EXTREME_VALUE,7
9,products.csv,weight_kg,NEGATIVE_VALUE,10


Numeric review records: 3054
Saved to reports/numeric_records_for_review.csv


## 22. Standardize Equivalent Category Labels

Review remaining category variants after trimming whitespace and normalizing case. Apply explicit mappings only when labels have the same business meaning, and save a mapping log so each standardization is traceable.

In [35]:

review_columns = {
    "orders.csv": ["order_status", "discount_code"],
    "payments.csv": ["payment_method", "payment_status"],
    "products.csv": ["category", "sub_category", "brand"],
    "sellers.csv": ["seller_tier"],
    "shipping.csv": ["courier", "delivery_status"],
    "returns.csv": ["return_status", "return_reason"]
}

for filename, columns in review_columns.items():
    df = load_processed(filename)

    for col in columns:
        if col not in df.columns:
            continue

        print(f"\n--- {filename} | {col} ---")
        print(
            df[col]
            .fillna("<MISSING>")
            .astype(str)
            .str.strip()
            .value_counts(dropna=False)
            .to_string()
        )



--- orders.csv | order_status ---
order_status
Delivered     125703
Cancelled      14525
Completed      13999
Returned        2619
In Transit      1169
Return           208
In-Transit        96
Shipped           76

--- orders.csv | discount_code ---
discount_code
No_Discount    108765
Save15           8609
Save10           8118
Save20           7494
Save40           6507
Save30           5765
Save5            5614
Save25           4553
Save50           2970

--- payments.csv | payment_method ---
payment_method
Cod                  73548
Card                 24738
Jazzcash             19087
Easypaisa            17833
Cash On Delivery     10591
Bank Transfer        10041
Credit/Debit Card     1845
Debit Card            1744
Jazz Cash             1741
Easy Paisa            1620
Bank Trf               960

--- payments.csv | payment_status ---
payment_status
Paid         129517
Cancelled     11151
Complete       9279
Success        9242
Refunded       2857
Canceled        761
Pending    

## 23. Validate Numeric Values and Record Quality Issues

Investigate negative quantities, prices, payment amounts, product weights, and shipping fees, as well as unusually large values. Statistical outliers are not automatically errors. Corrections must be supported by business rules or related records; unresolved issues must be flagged or quarantined with reason codes.

In [36]:

issue_examples = (
    numeric_issue_df
    .groupby(["source_file", "issue_column", "reason_code"], dropna=False)
    .head(5)
)

display(issue_examples)

print("Total issue records:", len(numeric_issue_df))
print(
    "Unique source-row combinations:",
    numeric_issue_df[
        ["source_file", "source_row_index"]
    ].drop_duplicates().shape[0]
)


,order_id,customer_id,order_date,order_status,ship_city_id,order_total,discount_code,channel,source_file,source_row_index,...,courier,shipped_date,delivered_date,shipping_fee,delivery_status,seller_name,join_date,seller_tier,city_id,rating
0,O0120349,C014012,2025-03-16,Delivered,56.0,9999999,No_Discount,App,orders.csv,144,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,O0114255,C007109,2025-02-28,Delivered,319.0,-21979,Save10,Mobile Web,orders.csv,325,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,O0003939,C028607,2024-01-22,Completed,347.0,9999999,No_Discount,App,orders.csv,2035,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,O0034353,C000951,2024-06-01,Completed,434.0,-8680,Save40,Web,orders.csv,2542,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,O0046326,C001588,2024-07-14,Delivered,450.0,-3445,No_Discount,Mweb,orders.csv,2655,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3044,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sellers.csv,4,...,NaN,NaN,NaN,NaN,NaN,Metro Traders 88,,Bronze,614,11.0
3045,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sellers.csv,14,...,NaN,NaN,NaN,NaN,NaN,Habib Bazaar 58,2023-04-11,Unknown,450,11.0
3046,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sellers.csv,18,...,NaN,NaN,NaN,NaN,NaN,Prime Mart 22,2020-07-17,Silver,31,11.0
3047,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,sellers.csv,42,...,NaN,NaN,NaN,NaN,NaN,Star Enterprises 32,2021-04-30,Silver,374,11.0


Total issue records: 3054
Unique source-row combinations: 3053


## 24. Normalize Confirmed Status Variants

Standardize confirmed spelling, spacing, and hyphenation variants in status fields. Preserve distinctions between different order and delivery states. Record the number of changed values for auditability.

In [37]:

status_mappings = {
    "orders.csv": {
        "order_status": {
            "in-transit": "In Transit",
            "return": "Returned",
            "delivered": "Delivered",
            "cancelled": "Cancelled",
            "completed": "Completed",
            "shipped": "Shipped",
        }
    },
    "shipping.csv": {
        "delivery_status": {
            "in-transit": "In Transit",
            "returned to seller": "Returned To Seller",
            "rto": "Returned To Seller",
            "delivered": "Delivered",
            "shipped": "Shipped",
        }
    }
}

status_change_log = []

for filename, columns in status_mappings.items():
    df = load_processed(filename)

    for col, mapping in columns.items():
        if col not in df.columns:
            continue

        original = df[col].copy()
        normalized = original.astype("string").str.strip()
        lookup = {k.casefold(): v for k, v in mapping.items()}

        df[col] = normalized.map(
            lambda value: lookup.get(value.casefold(), value)
            if pd.notna(value) else value
        )

        changed = int((original.fillna("<NA>") != df[col].fillna("<NA>")).sum())

        status_change_log.append({
            "file": filename,
            "column": col,
            "values_changed": changed
        })

    df.to_csv(PROCESSED_DIR / filename, index=False)

status_change_log_df = pd.DataFrame(status_change_log)
display(status_change_log_df)

status_change_log_df.to_csv(
    REPORT_DIR / "status_normalization_log.csv",
    index=False
)


,file,column,values_changed
0,orders.csv,order_status,304
1,shipping.csv,delivery_status,567


## 25. Validate Status Normalization

Confirm the final frequency distribution of normalized status fields and check that the intended variants no longer remain.

In [38]:

for filename, columns in {
    "orders.csv": ["order_status"],
    "shipping.csv": ["delivery_status"]
}.items():
    df = load_processed(filename)

    for col in columns:
        print(f"\n{filename} — {col}")
        display(
            df[col]
            .fillna("<MISSING>")
            .value_counts()
            .rename_axis(col)
            .reset_index(name="records")
        )



orders.csv — order_status


,order_status,records
0,Delivered,125703
1,Cancelled,14525
2,Completed,13999
3,Returned,2827
4,In Transit,1265
5,Shipped,76



shipping.csv — delivery_status


,delivery_status,records
0,Delivered,120611
1,Deliverd,7239
2,Completed,7139
3,Complete,7093
4,Returned To Seller,5370
5,In Transit,1268
6,Return To Seller,492
7,Shipped,74


## 26. Resolve Remaining Delivery-Status Variants

Standardize confirmed spelling variants in the delivery-status field while preserving distinct business states. Record the number of changes and validate the resulting labels.

In [39]:

shipping = load_processed("shipping.csv")

delivery_status_mapping = {
    "deliverd": "Delivered",
    "return to seller": "Returned To Seller",
}

original_status = shipping["delivery_status"].copy()
normalized_status = original_status.astype("string").str.strip()

lookup = {
    key.casefold(): value
    for key, value in delivery_status_mapping.items()
}

shipping["delivery_status"] = normalized_status.map(
    lambda value: lookup.get(value.casefold(), value)
    if pd.notna(value) else value
)

changed_count = int(
    (
        original_status.fillna("<NA>")
        != shipping["delivery_status"].fillna("<NA>")
    ).sum()
)

shipping.to_csv(PROCESSED_DIR / "shipping.csv", index=False)

print("Delivery-status values standardized:", changed_count)
print("\nFinal delivery-status distribution:")

display(
    shipping["delivery_status"]
    .fillna("<MISSING>")
    .value_counts()
    .rename_axis("delivery_status")
    .reset_index(name="records")
)

pd.DataFrame([{
    "file": "shipping.csv",
    "column": "delivery_status",
    "values_changed": changed_count,
    "mapping_applied": "Deliverd -> Delivered; Return To Seller -> Returned To Seller"
}]).to_csv(
    REPORT_DIR / "delivery_status_normalization_log.csv",
    index=False
)


Delivery-status values standardized: 7731

Final delivery-status distribution:


,delivery_status,records
0,Delivered,127850
1,Completed,7139
2,Complete,7093
3,Returned To Seller,5862
4,In Transit,1268
5,Shipped,74


## 27. Validate Delivery-Status Standardization

Check that the known spelling variants no longer remain and that the processed shipping row count has not changed.

In [40]:

shipping_check = load_processed("shipping.csv")
status_values = set(
    shipping_check["delivery_status"].dropna().astype(str).str.strip()
)

print("Shipping rows:", len(shipping_check))
print("Remaining 'Deliverd':", "Deliverd" in status_values)
print("Remaining 'Return To Seller':", "Return To Seller" in status_values)
print("Delivery-status missing values:", shipping_check["delivery_status"].isna().sum())


Shipping rows: 149286
Remaining 'Deliverd': False
Remaining 'Return To Seller': False
Delivery-status missing values: 0


## 28. Review Remaining Status Labels

Inspect the remaining distinct order and delivery statuses to identify equivalent labels without merging different business states. Apply only mappings supported by the observed values and the challenge rules.

In [41]:

for filename, column in [
    ("orders.csv", "order_status"),
    ("shipping.csv", "delivery_status"),
    ("returns.csv", "return_status"),
    ("payments.csv", "payment_status"),
]:
    df = load_processed(filename)

    print(f"\n--- {filename}: {column} ---")
    display(
        df[column]
        .fillna("<MISSING>")
        .value_counts()
        .rename_axis(column)
        .reset_index(name="records")
    )



--- orders.csv: order_status ---


,order_status,records
0,Delivered,125703
1,Cancelled,14525
2,Completed,13999
3,Returned,2827
4,In Transit,1265
5,Shipped,76



--- shipping.csv: delivery_status ---


,delivery_status,records
0,Delivered,127850
1,Completed,7139
2,Complete,7093
3,Returned To Seller,5862
4,In Transit,1268
5,Shipped,74



--- returns.csv: return_status ---


,return_status,records
0,Refunded,10461
1,Approved,3146
2,Rejected,1152
3,Refund,990
4,Declined,163



--- payments.csv: payment_status ---


,payment_status,records
0,Paid,129517
1,Cancelled,11151
2,Complete,9279
3,Success,9242
4,Refunded,2857
5,Canceled,761
6,Pending,690
7,Refund,251


## 29. Standardize Confirmed Status Variants

Map confirmed equivalent labels to one canonical value while preserving distinct business outcomes. In particular, rejected returns must remain distinct from valid returns, as required by the challenge brief. Log every change for traceability.

In [42]:

status_mappings = {
    "shipping.csv": {
        "delivery_status": {
            "complete": "Completed",
        }
    },
    "returns.csv": {
        "return_status": {
            "refund": "Refunded",
        }
    },
    "payments.csv": {
        "payment_status": {
            "complete": "Completed",
            "canceled": "Cancelled",
        }
    }
}

status_change_log = []

for filename, columns in status_mappings.items():
    df = load_processed(filename)

    for col, mapping in columns.items():
        original = df[col].copy()
        normalized = original.astype("string").str.strip()
        lookup = {key.casefold(): value for key, value in mapping.items()}

        df[col] = normalized.map(
            lambda value: lookup.get(value.casefold(), value)
            if pd.notna(value) else value
        )

        changed = int(
            (
                original.fillna("<NA>")
                != df[col].fillna("<NA>")
            ).sum()
        )

        status_change_log.append({
            "file": filename,
            "column": col,
            "values_changed": changed
        })

    df.to_csv(PROCESSED_DIR / filename, index=False)

status_change_log_df = pd.DataFrame(status_change_log)
display(status_change_log_df)

status_change_log_df.to_csv(
    REPORT_DIR / "additional_status_normalization_log.csv",
    index=False
)


,file,column,values_changed
0,shipping.csv,delivery_status,7093
1,returns.csv,return_status,990
2,payments.csv,payment_status,10040


## 30. Validate Standardized Status Fields

Review the final status distributions and verify that the known equivalent labels have been consolidated without changing row counts.

In [43]:

for filename, column in [
    ("shipping.csv", "delivery_status"),
    ("returns.csv", "return_status"),
    ("payments.csv", "payment_status"),
]:
    df = load_processed(filename)

    print(f"\n--- {filename}: {column} ---")
    display(
        df[column]
        .fillna("<MISSING>")
        .value_counts()
        .rename_axis(column)
        .reset_index(name="records")
    )



--- shipping.csv: delivery_status ---


,delivery_status,records
0,Delivered,127850
1,Completed,14232
2,Returned To Seller,5862
3,In Transit,1268
4,Shipped,74



--- returns.csv: return_status ---


,return_status,records
0,Refunded,11451
1,Approved,3146
2,Rejected,1152
3,Declined,163



--- payments.csv: payment_status ---


,payment_status,records
0,Paid,129517
1,Cancelled,11912
2,Completed,9279
3,Success,9242
4,Refunded,2857
5,Pending,690
6,Refund,251


## 31. Numeric Data Cleaning and Validation

Validate numeric fields against business rules and the expected data types. Negative or implausible values will be investigated before correction. Unresolved records will be preserved with reason codes; statistical outliers will not be removed solely because they fall outside an IQR threshold.

In [44]:

numeric_columns = {
    "orders.csv": ["order_total"],
    "order_items.csv": ["quantity", "unit_price", "discount_amount"],
    "payments.csv": ["amount"],
    "products.csv": ["cost_price", "weight_kg"],
    "returns.csv": ["refund_amount"],
    "shipping.csv": ["shipping_fee"],
    "sellers.csv": ["rating"],
}

negative_summary = []

for filename, columns in numeric_columns.items():
    df = load_processed(filename)

    for col in columns:
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col], errors="coerce")
        negative_mask = values < 0

        negative_summary.append({
            "file": filename,
            "column": col,
            "negative_records": int(negative_mask.sum()),
            "missing_records": int(values.isna().sum()),
            "minimum": values.min(),
            "maximum": values.max()
        })

negative_summary_df = pd.DataFrame(negative_summary)
display(negative_summary_df)

negative_summary_df.to_csv(
    REPORT_DIR / "negative_value_summary.csv",
    index=False
)


,file,column,negative_records,missing_records,minimum,maximum
0,orders.csv,order_total,163,0,-78112.0,9999999.0
1,order_items.csv,quantity,307,965,-6.0,500.0
2,order_items.csv,unit_price,316,0,-41270.0,9999999.0
3,order_items.csv,discount_amount,0,31567,0.0,163062.0
4,payments.csv,amount,656,0,-143168.0,9999999.0
5,products.csv,cost_price,0,170,82.0,83205.0
6,products.csv,weight_kg,10,296,-1.0,629.0
7,returns.csv,refund_amount,0,0,0.0,250592.0
8,shipping.csv,shipping_fee,155,9033,-2577.0,9999999.0
9,sellers.csv,rating,0,32,0.0,11.0


## 32. Review Negative-Value Examples

Display a small sample of negative values with their business identifiers. This allows us to distinguish invalid values from possible sign conventions or other explainable cases before making changes.

In [45]:

identifier_candidates = {
    "orders.csv": ["order_id"],
    "order_items.csv": ["order_item_id", "order_id", "product_id"],
    "payments.csv": ["payment_id", "order_id"],
    "products.csv": ["product_id"],
    "returns.csv": ["return_id", "order_id"],
    "shipping.csv": ["shipment_id", "order_id"],
    "sellers.csv": ["seller_id"],
}

for filename, columns in numeric_columns.items():
    df = load_processed(filename)

    for col in columns:
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col], errors="coerce")
        examples = df.loc[values < 0].copy()

        if examples.empty:
            continue

        id_cols = [
            c for c in identifier_candidates.get(filename, [])
            if c in examples.columns
        ]

        print(f"\n--- {filename}: {col} ---")
        display(
            examples[id_cols + [col]].head(8)
        )



--- orders.csv: order_total ---


,order_id,order_total
325,O0114255,-21979
2542,O0034353,-8680
2655,O0046326,-3445
3108,O0022010,-3467
3394,O0147331,-4643
5091,O0041493,-8501
6095,O0141234,-6485
9259,O0055384,-3526



--- order_items.csv: quantity ---


,order_item_id,order_id,product_id,quantity
87,OI00000088,O0000042,P02775,-4.0
248,OI00000249,O0000125,P00150,-5.0
2055,OI00002056,O0001051,P02866,-1.0
3384,OI00003385,O0001750,P02181,-2.0
3624,OI00003625,O0001878,P01570,-1.0
6244,OI00006245,O0003215,P01941,-2.0
6576,OI00006577,O0003375,P00930,-2.0
8185,OI00008186,O0004205,P01859,-1.0



--- order_items.csv: unit_price ---


...

,payment_id,order_id,amount
26,PAY00105717,O0102600,-1712
273,PAY00136735,O0132710,-2961
703,PAY00021594,O0020936,-1864
731,PAY00124362,O0120711,-5710
1011,PAY00136602,O0132579,-2595
1085,PAY00001959,O0001905,-3123
1164,PAY00136376,O0132358,-12370
1238,PAY00107986,O0104809,-6050



--- products.csv: weight_kg ---


,product_id,weight_kg
2,P00003,-1.0
490,P00491,-1.0
809,P00810,-1.0
1635,P01636,-1.0
1799,P01800,-1.0
2096,P02097,-1.0
2291,P02292,-1.0
2600,P02601,-1.0



--- shipping.csv: shipping_fee ---


,shipment_id,order_id,shipping_fee
665,SH00000671,O0000699,-299.0
2889,SH00002923,O0003055,-830.0
3458,SH00003501,O0003659,-291.0
4194,SH00004248,O0004438,-954.0
4600,SH00004659,O0004866,-396.0
5671,SH00005740,O0005999,-188.0
8136,SH00008241,O0008633,-754.0
8353,SH00008463,O0008872,-264.0


## 33. Investigate Extreme Monetary Values

Inspect unusually large monetary values alongside their record identifiers and related status fields. Repeated extreme values may indicate placeholders or data-entry errors, but they will not be changed without supporting evidence.

In [46]:

extreme_checks = {
    "orders.csv": ("order_total", 100000),
    "order_items.csv": ("unit_price", 100000),
    "payments.csv": ("amount", 100000),
    "shipping.csv": ("shipping_fee", 100000),
}

for filename, (col, threshold) in extreme_checks.items():
    df = load_processed(filename)

    values = pd.to_numeric(df[col], errors="coerce")
    extreme = df.loc[values >= threshold].copy()

    id_cols = [
        c for c in [
            "order_id", "order_item_id", "payment_id", "shipment_id",
            "order_status", "payment_status", "payment_method",
            "quantity", "unit_price", "order_total", "amount",
            "shipping_fee"
        ]
        if c in extreme.columns
    ]

    print(f"\n--- {filename}: {col} >= {threshold:,} ---")
    print("Matching records:", len(extreme))
    display(extreme[id_cols].head(10))



--- orders.csv: order_total >= 100,000 ---
Matching records: 432


,order_id,order_status,order_total
72,O0026563,Cancelled,110866
144,O0120349,Delivered,9999999
744,O0110056,Completed,183645
1063,O0120836,Completed,118477
1146,O0073561,Delivered,103350
2035,O0003939,Completed,9999999
3479,O0068174,Delivered,115564
3660,O0089727,Delivered,133381
3893,O0035277,Cancelled,9999999
4015,O0057125,Delivered,9999999



--- order_items.csv: unit_price >= 100,000 ---
Matching records: 523


,order_id,order_item_id,quantity,unit_price
636,O0000332,OI00000637,1.0,9999999.0
803,O0000421,OI00000804,2.0,272000.0
1769,O0000903,OI00001770,1.0,9999999.0
2120,O0001083,OI00002121,1.0,195100.0
2214,O0001127,OI00002215,1.0,9999999.0
3100,O0001607,OI00003101,2.0,161100.0
3305,O0001707,OI00003306,1.0,186100.0
4831,O0002508,OI00004832,1.0,9999999.0
6005,O0003094,OI00006006,3.0,9999999.0
6070,O0003127,OI00006071,1.0,9999999.0



--- payments.csv: amount >= 100,000 ---
Matching records: 449


,order_id,payment_id,payment_status,payment_method,amount
351,O0067752,PAY00069869,Paid,Cod,129993
404,O0112121,PAY00115522,Paid,Card,100097
1092,O0141562,PAY00145854,Paid,Cod,9999999
1242,O0120511,PAY00124154,Paid,Cash On Delivery,119439
1303,O0152401,PAY00156985,Cancelled,Cod,132474
1438,O0100002,PAY00103047,Paid,Card,100244
1917,O0009311,PAY00009592,Paid,Cod,104040
2931,O0076609,PAY00078975,Paid,Card,9999999
3113,O0116449,PAY00119988,Paid,Easypaisa,159809
3584,O0109860,PAY00113196,Paid,Cod,141206



--- shipping.csv: shipping_fee >= 100,000 ---
Matching records: 164


,order_id,shipment_id,shipping_fee
758,O0000795,SH00000765,9999999.0
1456,O0001538,SH00001472,9999999.0
2058,O0002177,SH00002082,9999999.0
2261,O0002388,SH00002287,9999999.0
2880,O0003046,SH00002914,9999999.0
3226,O0003413,SH00003265,9999999.0
3448,O0003648,SH00003491,9999999.0
3493,O0003697,SH00003536,9999999.0
3642,O0003854,SH00003686,9999999.0
5672,O0006000,SH00005741,9999999.0


## 33. Review Repeated Extreme Values

Identify repeated sentinel-like values across monetary fields. These values will be treated as suspected data-quality issues, not automatically corrected without supporting evidence.

In [47]:

sentinel_value = 9999999

sentinel_audit = []

for filename, columns in {
    "orders.csv": ["order_total"],
    "order_items.csv": ["unit_price", "discount_amount"],
    "payments.csv": ["amount"],
    "returns.csv": ["refund_amount"],
    "shipping.csv": ["shipping_fee"],
    "products.csv": ["cost_price", "weight_kg"],
}.items():
    df = load_processed(filename)

    for col in columns:
        values = pd.to_numeric(df[col], errors="coerce")
        sentinel_audit.append({
            "file": filename,
            "column": col,
            "sentinel_value": sentinel_value,
            "matching_records": int(values.eq(sentinel_value).sum()),
            "maximum_value": values.max(),
        })

sentinel_audit_df = pd.DataFrame(sentinel_audit)
display(sentinel_audit_df)

sentinel_audit_df.to_csv(
    REPORT_DIR / "sentinel_value_audit.csv",
    index=False
)


,file,column,sentinel_value,matching_records,maximum_value
0,orders.csv,order_total,9999999,145,9999999.0
1,order_items.csv,unit_price,9999999,305,9999999.0
2,order_items.csv,discount_amount,9999999,0,163062.0
3,payments.csv,amount,9999999,162,9999999.0
4,returns.csv,refund_amount,9999999,0,250592.0
5,shipping.csv,shipping_fee,9999999,164,9999999.0
6,products.csv,cost_price,9999999,0,83205.0
7,products.csv,weight_kg,9999999,0,629.0


## 34. Review Negative Numeric Values

Review negative quantities, monetary amounts and weights against related identifiers and statuses. Negative values will only be corrected or rejected when the business rules and supporting records justify that decision.

In [48]:

negative_checks = {
    "orders.csv": ["order_total"],
    "order_items.csv": ["quantity", "unit_price", "discount_amount"],
    "payments.csv": ["amount"],
    "products.csv": ["cost_price", "weight_kg"],
    "returns.csv": ["refund_amount"],
    "shipping.csv": ["shipping_fee"],
    "sellers.csv": ["rating"],
}

negative_rows = []

for filename, columns in negative_checks.items():
    df = load_processed(filename)

    for col in columns:
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col], errors="coerce")
        mask = values.lt(0)

        if mask.any():
            subset = df.loc[mask].copy()
            subset["issue_column"] = col
            subset["issue_value"] = values.loc[mask]
            subset["reason_code"] = "NEGATIVE_VALUE_REVIEW"
            subset["source_file"] = filename
            negative_rows.append(subset)

negative_review_df = (
    pd.concat(negative_rows, ignore_index=True, sort=False)
    if negative_rows else pd.DataFrame()
)

display(
    negative_review_df.head(15)
    if not negative_review_df.empty
    else "No negative values found."
)

negative_review_df.to_csv(
    REPORT_DIR / "negative_values_for_review.csv",
    index=False
)

print("Negative-value records flagged:", len(negative_review_df))
print("Review report saved.")


,order_id,customer_id,order_date,order_status,ship_city_id,order_total,discount_code,channel,issue_column,issue_value,...,brand,list_price,cost_price,weight_kg,shipment_id,courier,shipped_date,delivered_date,shipping_fee,delivery_status
0,O0114255,C007109,2025-02-28,Delivered,319.0,-21979,Save10,Mobile Web,order_total,-21979.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,O0034353,C000951,2024-06-01,Completed,434.0,-8680,Save40,Web,order_total,-8680.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,O0046326,C001588,2024-07-14,Delivered,450.0,-3445,No_Discount,Mweb,order_total,-3445.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,O0022010,C036413,2024-04-08,Delivered,347.0,-3467,Save20,Mobile Web,order_total,-3467.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,O0147331,C031284,2025-05-31,Delivered,450.0,-4643,Save15,Web,order_total,-4643.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,O0041493,C004062,2024-06-24,Delivered,281.0,-8501,No_Discount,Web,order_total,-8501.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,O0141234,C034178,2025-05-13,Delivered,614.0,-6485,No_Discount,Web,order_total,-6485.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,O0055384,C018831,2024-08-17,Delivered,597.0,-3526,No_Discount,App,order_total,-3526.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,O0155078,C039641,2025-06-16,Cancelled,255.0,-1128,No_Discount,App,order_total,-1128.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,O0042398,C007578,2024-06-28,Delivered,614.0,-23268,No_Discount,Mobile Web,order_total,-23268.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Negative-value records flagged: 1607
Review report saved.


## 35. Apply Documented Numeric Cleaning Rules

Apply fixes only to values supported by the challenge rules or corroborating records. Preserve unresolved cases in review or rejection outputs, and record before-and-after counts for every change.

## 36. Quarantine Suspected Sentinel Values

The numeric audit identified the repeated value `9,999,999` in several monetary fields. Because it is not a valid-looking business amount and appears repeatedly across datasets, affected records will be quarantined for review with a reason code. Raw files remain unchanged, and every affected record is preserved in the rejection output.

In [49]:

sentinel_value = 9999999

sentinel_columns = {
    "orders.csv": ["order_total"],
    "order_items.csv": ["unit_price"],
    "payments.csv": ["amount"],
    "shipping.csv": ["shipping_fee"],
}

sentinel_rejects = []

for filename, columns in sentinel_columns.items():
    df = load_processed(filename).copy()

    for col in columns:
        values = pd.to_numeric(df[col], errors="coerce")
        mask = values.eq(sentinel_value)

        if mask.any():
            rejected = df.loc[mask].copy()
            rejected["source_file"] = filename
            rejected["source_row_index"] = rejected.index
            rejected["issue_column"] = col
            rejected["issue_value"] = values.loc[mask]
            rejected["reason_code"] = "SUSPECT_SENTINEL_VALUE"
            sentinel_rejects.append(rejected)

sentinel_rejects_df = (
    pd.concat(sentinel_rejects, ignore_index=True, sort=False)
    if sentinel_rejects else pd.DataFrame()
)

display(
    sentinel_rejects_df[
        ["source_file", "source_row_index", "issue_column",
         "issue_value", "reason_code"]
    ].groupby(
        ["source_file", "issue_column", "issue_value", "reason_code"],
        dropna=False
    ).size().reset_index(name="records")
)

sentinel_rejects_df.to_csv(
    REJECT_DIR / "sentinel_value_rejects.csv",
    index=False
)

print("Affected records saved:", len(sentinel_rejects_df))
print("Rejection file:", REJECT_DIR / "sentinel_value_rejects.csv")


,source_file,issue_column,issue_value,reason_code,records
0,order_items.csv,unit_price,9999999.0,SUSPECT_SENTINEL_VALUE,305
1,orders.csv,order_total,9999999.0,SUSPECT_SENTINEL_VALUE,145
2,payments.csv,amount,9999999.0,SUSPECT_SENTINEL_VALUE,162
3,shipping.csv,shipping_fee,9999999.0,SUSPECT_SENTINEL_VALUE,164


Affected records saved: 776
Rejection file: c:\Users\LENOVO\Documents\CodeConnect26_DataAnalytics\data\rejected\sentinel_value_rejects.csv


## 37. Quarantine Suspected Sentinel Records

Remove records containing the suspected sentinel value from the working processed datasets and preserve them in a rejection file with reason codes. Reconcile the original processed row counts against retained and quarantined records. Raw input files remain unchanged.

In [50]:

sentinel_columns = {
    "orders.csv": ["order_total"],
    "order_items.csv": ["unit_price"],
    "payments.csv": ["amount"],
    "shipping.csv": ["shipping_fee"],
}

sentinel_quarantine_log = []

for filename, columns in sentinel_columns.items():
    path = PROCESSED_DIR / filename
    df = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)

    row_mask = pd.Series(False, index=df.index)
    reason_by_row = {}

    for col in columns:
        values = pd.to_numeric(df[col], errors="coerce")
        mask = values.eq(9999999)

        for idx in df.index[mask]:
            row_mask.loc[idx] = True
            reason_by_row.setdefault(idx, []).append(
                f"SUSPECT_SENTINEL_VALUE:{col}"
            )

    rejected = df.loc[row_mask].copy()
    rejected["source_file"] = filename
    rejected["source_row_index"] = rejected.index
    rejected["reason_code"] = [
        ";".join(reason_by_row[idx]) for idx in rejected.index
    ]

    retained = df.loc[~row_mask].copy()
    rejected.to_csv(
        REJECT_DIR / f"sentinel_{filename}", index=False
    )
    retained.to_csv(path, index=False)

    sentinel_quarantine_log.append({
        "file": filename,
        "rows_before": len(df),
        "rows_quarantined": int(row_mask.sum()),
        "rows_after": len(retained),
        "reconciles": len(retained) + int(row_mask.sum()) == len(df),
    })

sentinel_quarantine_df = pd.DataFrame(sentinel_quarantine_log)
display(sentinel_quarantine_df)

sentinel_quarantine_df.to_csv(
    REPORT_DIR / "sentinel_quarantine_log.csv", index=False
)

print("All row counts reconcile:",
      sentinel_quarantine_df["reconciles"].all())


,file,rows_before,rows_quarantined,rows_after,reconciles
0,orders.csv,158395,145,158250,True
1,order_items.csv,315143,305,314838,True
2,payments.csv,163748,162,163586,True
3,shipping.csv,149286,164,149122,True


All row counts reconcile: True


## 38. Reassess Numeric Quality After Quarantine

Recalculate negative values, missing values and numeric ranges after sentinel quarantine. This provides a current baseline for remaining numeric cleaning decisions.

In [51]:

numeric_fields = {
    "orders.csv": ["order_total"],
    "order_items.csv": ["quantity", "unit_price", "discount_amount"],
    "payments.csv": ["amount"],
    "products.csv": ["cost_price", "weight_kg"],
    "returns.csv": ["refund_amount"],
    "shipping.csv": ["shipping_fee"],
    "sellers.csv": ["rating"],
}

numeric_quality = []

for filename, columns in numeric_fields.items():
    df = load_processed(filename)

    for col in columns:
        if col not in df.columns:
            continue

        values = pd.to_numeric(df[col], errors="coerce")

        numeric_quality.append({
            "file": filename,
            "column": col,
            "rows": len(df),
            "missing_or_unparseable": int(values.isna().sum()),
            "negative_values": int(values.lt(0).sum()),
            "minimum": values.min(),
            "maximum": values.max(),
        })

numeric_quality_df = pd.DataFrame(numeric_quality)
display(numeric_quality_df)

numeric_quality_df.to_csv(
    REPORT_DIR / "numeric_quality_after_sentinel.csv",
    index=False
)


,file,column,rows,missing_or_unparseable,negative_values,minimum,maximum
0,orders.csv,order_total,158250,0,163,-78112.0,326150.0
1,order_items.csv,quantity,314838,965,307,-6.0,500.0
2,order_items.csv,unit_price,314838,0,316,-41270.0,3801600.0
3,order_items.csv,discount_amount,314838,31535,0,0.0,163062.0
4,payments.csv,amount,163586,0,656,-143168.0,326150.0
5,products.csv,cost_price,3200,170,0,82.0,83205.0
6,products.csv,weight_kg,3200,296,10,-1.0,629.0
7,returns.csv,refund_amount,15912,0,0,0.0,250592.0
8,shipping.csv,shipping_fee,149122,9033,155,-2577.0,6127.0
9,sellers.csv,rating,416,32,0,0.0,11.0


## 39. Finalize Missing-Value Handling Decisions

Document the treatment of missing fields by business meaning. Do not invent customer details, dates, costs, weights or delivery charges. Retain optional missing information, use explicit Unknown members for warehouse dimensions where required, and flag or reject missing values that prevent reliable calculations.

## 40. Normalize Quantities and Flag Invalid Numeric Records

Convert recognizable number words in quantity fields into numeric values. Flag missing or non-positive quantities, negative prices, negative weights, negative order totals and negative shipping fees for documented review. Negative payment amounts are retained for status-aware investigation because they may represent reversals or adjustments.

In [52]:

# Normalize common number words in order-item quantities.
items_path = PROCESSED_DIR / "order_items.csv"
items = pd.read_csv(items_path, encoding="utf-8-sig", low_memory=False)

quantity_words = {
    "zero": "0",
    "one": "1",
    "two": "2",
    "three": "3",
    "four": "4",
    "five": "5",
    "six": "6",
    "seven": "7",
    "eight": "8",
    "nine": "9",
    "ten": "10",
}

raw_quantity = items["quantity"].astype("string").str.strip()
normalized_quantity = raw_quantity.str.lower().replace(quantity_words)
numeric_quantity = pd.to_numeric(normalized_quantity, errors="coerce")

quantity_fix_count = int(
    (raw_quantity.notna() & numeric_quantity.notna()
     & raw_quantity.ne(normalized_quantity)).sum()
)

items["quantity"] = numeric_quantity
items.to_csv(items_path, index=False)

print("Quantity values converted from words:", quantity_fix_count)
print("Missing/unparseable quantities remaining:", int(numeric_quantity.isna().sum()))
print("Zero or negative quantities:", int(numeric_quantity.le(0).sum()))
print("Positive quantities:", int(numeric_quantity.gt(0).sum()))

# Review numeric issues without silently deleting records.
review_rules = {
    "orders.csv": {
        "order_total": ("NEGATIVE_ORDER_TOTAL", lambda s: s < 0)
    },
    "order_items.csv": {
        "quantity": ("MISSING_OR_UNPARSEABLE_QUANTITY", lambda s: s.isna()),
        "unit_price": ("NEGATIVE_UNIT_PRICE", lambda s: s < 0),
    },
    "products.csv": {
        "weight_kg": ("NEGATIVE_PRODUCT_WEIGHT", lambda s: s < 0),
    },
    "shipping.csv": {
        "shipping_fee": ("NEGATIVE_SHIPPING_FEE", lambda s: s < 0),
    },
}

numeric_review_rows = []

for filename, rules in review_rules.items():
    df = load_processed(filename)

    for col, (reason, rule) in rules.items():
        values = pd.to_numeric(df[col], errors="coerce")
        mask = rule(values)

        if mask.any():
            part = df.loc[mask].copy()
            part["source_file"] = filename
            part["source_row_index"] = part.index
            part["issue_column"] = col
            part["issue_value"] = values.loc[mask]
            part["reason_code"] = reason
            numeric_review_rows.append(part)

numeric_review_df = (
    pd.concat(numeric_review_rows, ignore_index=True, sort=False)
    if numeric_review_rows else pd.DataFrame()
)

numeric_review_df.to_csv(
    REPORT_DIR / "numeric_invalid_values_for_review.csv",
    index=False
)

if not numeric_review_df.empty:
    display(
        numeric_review_df.groupby(
            ["source_file", "issue_column", "reason_code"],
            dropna=False
        ).size().reset_index(name="records")
    )

print("Numeric review records:", len(numeric_review_df))
print("Saved: reports/numeric_invalid_values_for_review.csv")


Quantity values converted from words: 0
Missing/unparseable quantities remaining: 965
Zero or negative quantities: 307
Positive quantities: 313566


,source_file,issue_column,reason_code,records
0,order_items.csv,quantity,MISSING_OR_UNPARSEABLE_QUANTITY,965
1,order_items.csv,unit_price,NEGATIVE_UNIT_PRICE,316
2,orders.csv,order_total,NEGATIVE_ORDER_TOTAL,163
3,products.csv,weight_kg,NEGATIVE_PRODUCT_WEIGHT,10
4,shipping.csv,shipping_fee,NEGATIVE_SHIPPING_FEE,155


Numeric review records: 1609
Saved: reports/numeric_invalid_values_for_review.csv


## 41. Cleaning Progress and Outstanding Decisions

Record completed transformations separately from unresolved review cases. No issue is considered resolved merely because it has been detected or reported. Remaining cases must be repaired with evidence, retained with an explicit limitation, mapped to an Unknown warehouse member where appropriate, or quarantined with a reason code. Before-and-after counts must reconcile.

In [53]:

checkpoint = pd.DataFrame([
    ("Exact duplicate rows", "Completed", "duplicate_removal_log.csv"),
    ("Invalid dates", "Quarantined", "date rejects output"),
    ("Suspected sentinel values", "Quarantined", "sentinel_quarantine_log.csv"),
    ("Category/status standardization", "Partially completed", "category normalization reports"),
    ("Numeric invalid values", "Review in progress", "numeric_invalid_values_for_review.csv"),
    ("Missing values", "Decisions still required", "column-level cleaning decisions"),
    ("Duplicate entity resolution", "Candidates need evidence review", "duplicate entity candidates report"),
    ("Broken references", "Audit completed; repairs pending", "reference audit reports"),
], columns=["requirement", "status", "evidence"])

display(checkpoint)
checkpoint.to_csv(REPORT_DIR / "cleaning_progress_checkpoint.csv", index=False)


,requirement,status,evidence
0,Exact duplicate rows,Completed,duplicate_removal_log.csv
1,Invalid dates,Quarantined,date rejects output
2,Suspected sentinel values,Quarantined,sentinel_quarantine_log.csv
3,Category/status standardization,Partially completed,category normalization reports
4,Numeric invalid values,Review in progress,numeric_invalid_values_for_review.csv
5,Missing values,Decisions still required,column-level cleaning decisions
6,Duplicate entity resolution,Candidates need evidence review,duplicate entity candidates report
7,Broken references,Audit completed; repairs pending,reference audit reports


## 42. Resolve Clearly Invalid Numeric Records

Quarantine order-item rows with missing or non-positive quantities or negative unit prices, using documented reason codes. For negative order totals, product weights and shipping fees, preserve the original records and flag the affected fields for review rather than deleting entire business entities. This prevents avoidable loss of valid relationships and supports later warehouse reconciliation.

In [54]:

# Quarantine invalid order-item rows only.
items_path = PROCESSED_DIR / "order_items.csv"
items = pd.read_csv(items_path, encoding="utf-8-sig", low_memory=False)

quantity = pd.to_numeric(items["quantity"], errors="coerce")
unit_price = pd.to_numeric(items["unit_price"], errors="coerce")

reason_map = pd.Series("", index=items.index, dtype="string")

reason_map.loc[quantity.isna()] = "MISSING_OR_UNPARSEABLE_QUANTITY"
reason_map.loc[quantity.le(0)] = (
    reason_map.loc[quantity.le(0)].fillna("").astype(str)
    .map(lambda x: (x + ";" if x else "") + "NON_POSITIVE_QUANTITY")
)
reason_map.loc[unit_price.lt(0)] = (
    reason_map.loc[unit_price.lt(0)].fillna("").astype(str)
    .map(lambda x: (x + ";" if x else "") + "NEGATIVE_UNIT_PRICE")
)

invalid_mask = reason_map.ne("")

# Keep the original rows and attach reason codes in the reject file.
rejected_items = items.loc[invalid_mask].copy()
rejected_items["source_file"] = "order_items.csv"
rejected_items["source_row_index"] = rejected_items.index
rejected_items["reason_code"] = reason_map.loc[invalid_mask].values

retained_items = items.loc[~invalid_mask].copy()

rejected_items.to_csv(
    REJECT_DIR / "invalid_order_items.csv", index=False
)
retained_items.to_csv(items_path, index=False)

numeric_row_log = pd.DataFrame([{
    "file": "order_items.csv",
    "rows_before": len(items),
    "rows_quarantined": int(invalid_mask.sum()),
    "rows_after": len(retained_items),
    "reconciles": len(retained_items) + int(invalid_mask.sum()) == len(items),
}])

display(numeric_row_log)
numeric_row_log.to_csv(
    REPORT_DIR / "invalid_order_items_quarantine_log.csv",
    index=False
)

print("Quarantined order-item rows:", int(invalid_mask.sum()))
print("Remaining order-item rows:", len(retained_items))
print("Counts reconcile:", len(retained_items) + int(invalid_mask.sum()) == len(items))


,file,rows_before,rows_quarantined,rows_after,reconciles
0,order_items.csv,314838,1588,313250,True


Quarantined order-item rows: 1588
Remaining order-item rows: 313250
Counts reconcile: True


## 43. Flag Remaining Numeric Issues

Preserve order, product and shipping records that contain invalid numeric fields, but record the affected field, original value and reason code. These flags will guide warehouse calculations and prevent implausible values from being treated as valid measures.

In [55]:

field_rules = {
    "orders.csv": {
        "order_total": ("NEGATIVE_ORDER_TOTAL", lambda s: s < 0),
    },
    "products.csv": {
        "weight_kg": ("NEGATIVE_PRODUCT_WEIGHT", lambda s: s < 0),
    },
    "shipping.csv": {
        "shipping_fee": ("NEGATIVE_SHIPPING_FEE", lambda s: s < 0),
    },
}

field_flags = []

for filename, rules in field_rules.items():
    df = load_processed(filename)

    for col, (reason, rule) in rules.items():
        values = pd.to_numeric(df[col], errors="coerce")
        mask = rule(values)

        if mask.any():
            flagged = df.loc[mask].copy()
            flagged["source_file"] = filename
            flagged["source_row_index"] = flagged.index
            flagged["issue_column"] = col
            flagged["original_value"] = values.loc[mask]
            flagged["reason_code"] = reason
            field_flags.append(flagged)

field_flags_df = (
    pd.concat(field_flags, ignore_index=True, sort=False)
    if field_flags else pd.DataFrame()
)

field_flags_df.to_csv(
    REJECT_DIR / "numeric_field_flags.csv", index=False
)

if not field_flags_df.empty:
    display(
        field_flags_df.groupby(
            ["source_file", "issue_column", "reason_code"],
            dropna=False
        ).size().reset_index(name="records")
    )

print("Flagged field-level records:", len(field_flags_df))
print("Saved: data/rejected/numeric_field_flags.csv")


,source_file,issue_column,reason_code,records
0,orders.csv,order_total,NEGATIVE_ORDER_TOTAL,163
1,products.csv,weight_kg,NEGATIVE_PRODUCT_WEIGHT,10
2,shipping.csv,shipping_fee,NEGATIVE_SHIPPING_FEE,155


Flagged field-level records: 328
Saved: data/rejected/numeric_field_flags.csv


## 44. Re-audit References After Numeric Quarantine

Recheck foreign-key relationships after removing invalid order-item rows. Report missing and unmatched references before applying the documented Unknown-member or quarantine strategy. Do not infer identity from names alone.

In [56]:

customers = load_processed("customers.csv")
products = load_processed("products.csv")
locations = load_processed("locations.csv")
orders = load_processed("orders.csv")
items = load_processed("order_items.csv")
payments = load_processed("payments.csv")
shipping = load_processed("shipping.csv")

reference_pairs = [
    ("orders.customer_id", orders, "customer_id", customers, "customer_id"),
    ("orders.ship_city_id", orders, "ship_city_id", locations, "city_id"),
    ("order_items.order_id", items, "order_id", orders, "order_id"),
    ("order_items.product_id", items, "product_id", products, "product_id"),
    ("payments.order_id", payments, "order_id", orders, "order_id"),
    ("shipping.order_id", shipping, "order_id", orders, "order_id"),
]

reference_results = []

for label, child, child_col, parent, parent_col in reference_pairs:
    child_keys = child[child_col].astype("string").str.strip()
    parent_keys = set(
        parent[parent_col].dropna().astype("string").str.strip()
    )

    missing = child_keys.isna() | child_keys.eq("")
    unmatched = ~missing & ~child_keys.isin(parent_keys)

    reference_results.append({
        "relationship": label,
        "rows_checked": len(child),
        "missing_keys": int(missing.sum()),
        "unmatched_keys": int(unmatched.sum()),
        "valid_keys": int((~missing & ~unmatched).sum()),
    })

reference_results_df = pd.DataFrame(reference_results)
display(reference_results_df)

reference_results_df.to_csv(
    REPORT_DIR / "reference_audit_after_numeric_quarantine.csv",
    index=False
)


,relationship,rows_checked,missing_keys,unmatched_keys,valid_keys
0,orders.customer_id,158250,0,318,157932
1,orders.ship_city_id,158250,6322,151928,0
2,order_items.order_id,313250,0,3465,309785
3,order_items.product_id,313250,0,925,312325
4,payments.order_id,163586,0,1781,161805
5,shipping.order_id,149122,0,1645,147477



## Final Cleaning Summary and Handoff

The cleaning workflow was implemented using Python, with the original raw CSV files preserved. Exact duplicates, invalid dates, suspected sentinel values, inconsistent categorical labels, and numeric anomalies were handled through documented transformations, review flags, or reason-coded rejection files.

Validation reported successful row-count reconciliation across all nine datasets, no remaining exact duplicate rows, no duplicate primary keys, and no unparseable non-null numeric values. A consolidated quality report, cleaning decision log, missing-value decisions, category mappings, and reference-exception reports were generated.

Some data-quality exceptions remain unresolved. These include broken references, records linked to quarantined parent records, missing values, and flagged customer registration dates. They have not been silently discarded or treated as fully resolved.

The processed CSVs are ready for unconstrained SQL Server staging. Foreign-key-constrained loading and strict date conversion will follow documented exception-handling decisions during warehouse preparation. The next phase is SQL Server staging, followed by the required star schema and Airflow orchestration.
